# Sudoku Digit CNN - Colab Training Notebook

End-to-end training for the digit-recognition CNN used by the image-based Sudoku solver
(CPECOG1 project).

**What this notebook does**

1.  Gets a dataset of **real photos of sudoku grids** with per-image ground truth
    (`mexwell/sudoku-image-dataset` on Kaggle, or its identical public GitHub mirror).
2.  Extracts **labeled digit crops** from those photos (perspective warp + cell
    preprocessing with adaptive thresholding).
3.  Trains on MNIST + EMNIST handwriting, curated print-font synthetic digits
    (all 1-9) and synthetic empty cells for class 0 - AND, optionally
    (`CONFIG["include_photo"]`), **real photo cells from
    Lexski/sudoku-image-recognition** under a **sealed 70/15/15 split** built
    with the same seed as the repo's `photo_data.py`. The TEST partition of
    that split is never loaded during training - it is the official benchmark
    (section 13).
4.  Splits everything **70 / 15 / 15 (train / val / test), stratified by digit, seeded
    and reproducible**. The test slice is only touched once, at the very end.
5.  Trains the `DigitCNN` with per-epoch **checkpointing** (resume-safe), optionally
    **warm-started** from a synthetic checkpoint (`CONFIG["warm_start"]`).
6.  Evaluates on the held-out test split, fits a **temperature-scaling** factor on the
    validation split and exports `digit_cnn.pth` + `digit_cnn.pth.temperature.json`,
    the exact files the repo's `load_digit_model()` / `load_temperature()` expect.
7.  **Optional** end-to-end verification on your own sudoku photos (e.g. `cpecog.zip`).

**How to run:** `Runtime > Run all` (use a GPU runtime: `Runtime > Change runtime type > T4 GPU`).
The first section runs fine with zero credentials (GitHub mirror fallback).


In [ ]:
# --- 1. Setup: packages, device, seeds ----------------------------------------
# Colab ships torch, torchvision, cv2, matplotlib and PIL preinstalled (with GPU
# wheels on a GPU runtime) - only the kaggle CLI is missing. Do NOT pip install
# torch/torchvision/opencv here: it would replace the preinstalled GPU build.
!pip install -q kaggle

import os
import re
import json
import random
import glob

import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch:", torch.__version__, "| cv2:", cv2.__version__)

# fixed seeds -> reproducible partitions, augmentation and training
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 2. Configuration

Everything you usually want to change lives in this one dictionary: how the data is
obtained, how long to train, and where checkpoints go.


In [ ]:
# --- 2. Configuration ----------------------------------------------------------
CONFIG = dict(
    # how to obtain the real-photo dataset:
    #   "auto"   use Kaggle API if kaggle.json was uploaded, otherwise git clone the
    #            public GitHub mirror (no credentials needed)
    #   "kaggle" download mexwell/sudoku-image-dataset via the Kaggle API
    #   "github" git clone https://github.com/wichtounet/sudoku_dataset
    #   "upload" manually upload a zip of the dataset
    #   "drive"  use a folder on Google Drive (DATASET_DIR)
    data_mode="auto",
    kaggle_dataset="mexwell/sudoku-image-dataset",
    github_url="https://github.com/wichtounet/sudoku_dataset",
    data_dir="/content/data",
    drive_dataset_dir="/content/drive/MyDrive/sudoku_data",

    # digit-model training
    epochs=40,                      # hard cap; early stopping normally ends training sooner
    early_stop_patience=5,          # stop once val acc has not improved for this many epochs
    min_epochs=12,                  # ...but never before this (AdamW + ReduceLROnPlateau
                                    # needs a few flat epochs before the LR drops)
    batch_size=256,
    lr=3e-4,                        # AdamW lr (weight decay 1e-4, see section 10).
                                    # Use lr=1e-4 when warm-started (fine-tune mode).
    warm_start=None,                # optional .pth checkpoint to fine-tune from
                                    # (a synthetic-only model); keep lr=1e-4 then
    empty_per_class=15000,          # synthetic EMPTY-cell (class 0) samples - matches
                                    # the ~10k digit samples per class from MNIST+synth
    include_mnist=True,             # MNIST handwritten digits 1-9 (digit 0 dropped;
                                    # handwritten sudoku puzzles and pen-filled cells)
    include_emnist=True,            # EMNIST handwritten digits 1-9 (split='digits',
                                    # ~280k samples; digit 0 dropped). 0 = use all,
                                    # or set emnist_per_class to cap for faster runs
    emnist_per_class=0,
    emnist_root="/content/emnist",
    include_synth=True,             # curated print-font synthetic digits 1-9
                                    # (DejaVu/Arial/Times/Courier/... - no decorative
                                    # fonts) - unlimited clean printed digits
    synth_per_class=4000,
    mnist_per_class=0,              # 0 = full MNIST; cap for smoke runs (parity
                                    # with train_local.py --mnist-per-class)
    mnist_root="/content/mnist",
    include_photo=True,             # REAL photo cells (Lexski) in training - the
                                    # whole point of the capstone is the photo
                                    # domain, so this is ON by default. The sealed
                                    # 70/15/15 split is rebuilt here with the
                                    # SAME seed (42) and algorithm as photo_data.py,
                                    # so the partitions match the repo exactly. The
                                    # TEST partition is sealed - never loaded here.
                                    # Downloads the whole dataset (~100 MB) on first use.
    photo_hf_dir="/content/data",   # Lexski metadata + images root (data/<split>/)
    photo_split_seed=42,            # MUST stay 42 to match the repo's photo_splits.json
    ckpt_dir="checkpoints",
    drive_dir="/content/drive/MyDrive/CPECOG1",   # Google Drive output folder:
                                    # per-epoch checkpoints + the exported
                                    # digit_cnn.pth land here (auto-mounted by
                                    # the training cell) so a runtime reset
                                    # becomes a resume, not a restart

    # data partition (test gets the remainder: 15%)
    train_frac=0.7,
    val_frac=0.15,

    # labeled benchmark. Photo data may be used for TRAINING under the sealed
    # split (include_photo); the benchmark numbers on the SEALED test
    # partition (hf_full_test) describe photos the model never saw.
    hf_benchmark=True,              # benchmark on Lexski/sudoku-image-recognition
                                    # (1400 images, explicit train/val/test splits,
                                    # solved-cell ground truth + grid keypoints)
    hf_split="test",
    hf_sample_size=24,              # lightweight: download only a seeded sample of
                                    # images (no datasets lib, a few MB). Raise to
                                    # e.g. 200 for the full split.
    hf_full_test=False,             # True: benchmark the FULL sealed test partition
                                    # (210 puzzles, same names as the repo's
                                    # photo_splits.json - needs include_photo=True
                                    # or at least the metadata downloaded). This is
                                    # the OFFICIAL number for photo-trained models.
    labeled_eval_frac=0.5,          # wichtounet folder fallback: seeded half of the photos
    max_correction_cells=12,        # bound solver correction to the N least-confident cells

    # optional grid-detection-only check on unlabeled photos
    # (e.g. the 2620 JPEGs extracted from cpecog.zip)
    cpecog_dir=None,
    cpecog_eval_frac=0.2,
)


## 3. Dataset provisioning

The primary dataset is the Kaggle **Sudoku Image Dataset** (mexwell), which mirrors the
public `wichtounet/sudoku_dataset` repo: photos of sudoku grids, one `.dat` file per
photo with the ground-truth puzzle (0 = empty, 1-9 = digits), and
`outlines_sorted.csv` with the grid corners of every photo.

* **Kaggle API** gives the dataset as it exists on Kaggle; needs your `kaggle.json`.
* **GitHub mirror** (default in `auto` mode) needs nothing.


In [ ]:
# --- 3. Get the dataset ----------------------------------------------------------
os.makedirs(CONFIG["data_dir"], exist_ok=True)

def provision_dataset():
    mode = CONFIG["data_mode"]
    if mode in ("auto", "kaggle"):
        kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
        if os.path.exists(kaggle_json):
            print("kaggle.json found -> downloading via Kaggle API")
            ok = os.system(f"kaggle datasets download -d {CONFIG['kaggle_dataset']} "
                           f"--unzip -p {CONFIG['data_dir']}")
            if ok == 0:
                return
            print("Kaggle download failed, falling back to the GitHub mirror.")
        elif mode == "kaggle":
            print("kaggle.json not found. Upload it with the next cell, or set "
                  "data_mode='github' / 'upload' / 'drive'.")
            return
    if mode in ("auto", "github"):
        target = os.path.join(CONFIG["data_dir"], "sudoku_dataset")
        if not os.path.isdir(target):
            print("cloning public GitHub mirror (no credentials needed)...")
            os.system(f"git clone --depth 1 {CONFIG['github_url']} {target}")
        else:
            print("GitHub mirror already present.")
        return
    if mode == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        print("Put the dataset folder in", CONFIG["drive_dataset_dir"],
              "then re-run this cell (or set data_mode='github').")
        return
    if mode == "upload":
        from google.colab import files
        print("Upload the dataset zip now...")
        for name in files.upload():
            os.system(f"unzip -q -o {name} -d {CONFIG['data_dir']}")
        return
    raise ValueError("unknown data_mode: " + mode)

provision_dataset()


In [ ]:
# --- 3b. OPTIONAL: install your Kaggle API token --------------------------------
# Only needed for data_mode='kaggle'. Get the token at
# https://www.kaggle.com/settings -> API -> Create New Token, upload kaggle.json
# in this cell, then re-run the provisioning cell above.
# The upload prompt only appears when it is actually required, so a plain
# Runtime > Run all never blocks here.
if CONFIG["data_mode"] == "kaggle" and not os.path.exists(
        os.path.expanduser("~/.kaggle/kaggle.json")):
    from google.colab import files
    up = files.upload()
    if "kaggle.json" in up:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        os.system("cp kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json")
        print("kaggle.json installed - re-run the provisioning cell.")
else:
    print("skipped: only needed for data_mode='kaggle' without ~/.kaggle/kaggle.json")


## 4. Explore the dataset

Two quick helpers are introduced:

* `parse_dat` reads the ground-truth puzzle from a `.dat` file (2 header lines with
  camera info, then 9 lines of 9 space-separated digits, 0 = empty cell).
* `load_outlines` reads `outlines_sorted.csv` into `{filename: 4x2 corners}` where the
  corners are already ordered top-left, top-right, bottom-right, bottom-left.


In [ ]:
# --- 4. Explore the dataset ----------------------------------------------------
def parse_dat(path):
    """Read a .dat puzzle file -> 9x9 int array (0 = empty cell)."""
    lines = open(path, encoding="utf-8", errors="ignore").read().strip().splitlines()
    rows = []
    for line in lines:
        toks = line.split()
        if len(toks) >= 9 and all(t.isdigit() and int(t) <= 9 for t in toks[:9]):
            rows.append([int(t) for t in toks[:9]])
        elif len(line.strip()) == 9 and all(c.isdigit() for c in line.strip()):
            rows.append([int(c) for c in line.strip()])
        if len(rows) == 9:
            break
    assert len(rows) == 9, f"could not parse 9 rows from {path}"
    return np.array(rows, dtype=int)


def load_outlines(csv_path):
    """outlines_sorted.csv -> {filename: (absolute_image_path, 4x2 corners)}.

    The CSV is the master list: each row names the exact image file (relative
    to the CSV) and its four ordered corners, so we only ever crop the photo
    the outline was drawn on.
    """
    outlines = {}
    base_dir = os.path.dirname(os.path.abspath(csv_path))
    with open(csv_path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 9:
                continue
            rel = parts[0].replace("\\", "/").lstrip("./")
            img_path = os.path.normpath(os.path.join(base_dir, rel))
            try:
                pts = np.array([float(x) for x in parts[1:9]]).reshape(4, 2)
            except ValueError:
                continue
            outlines[os.path.basename(img_path)] = (img_path, pts)
    return outlines


def find_dataset_files(root):
    images = sorted(glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True))
    dat_files = sorted(glob.glob(os.path.join(root, "**", "*.dat"), recursive=True))
    csv_path = None
    for cand in glob.glob(os.path.join(root, "**", "outlines_sorted.csv"), recursive=True):
        csv_path = cand
        break
    return images, dat_files, csv_path


images, dat_files, csv_path = find_dataset_files(CONFIG["data_dir"])
print("images:", len(images), "| .dat files:", len(dat_files), "| outlines csv:", csv_path)


In [ ]:
# --- 4b. Look at one photo + its ground truth ---------------------------------
def show_montage(imgs, labels=None, n=16, title="", seed=0, classes=None):
    """Random, class-balanced preview (fixed seed). Without this, the montages
    show the first array entries, which are class-sorted and all the same class."""
    n = min(n, len(imgs))
    if n == 0:
        print("(nothing to show)")
        return
    rng = np.random.RandomState(seed)
    if classes is not None and len(np.unique(classes)) > 1:
        pick = []
        per = max(1, n // len(np.unique(classes)))
        for c in np.unique(classes):
            cand = np.where(classes == c)[0]
            rng.shuffle(cand)
            pick += list(cand[:per])
        idx = np.array(pick[:n])
        rng.shuffle(idx)
    else:
        idx = rng.permutation(len(imgs))[:n]
    n = min(n, len(idx))
    imgs = np.asarray(imgs)[idx]
    if labels is not None:
        labels = np.asarray(labels)[idx]
    cols = 8
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.6))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        axes[i].imshow(imgs[i], cmap="gray", vmin=0, vmax=1)
        axes[i].axis("off")
        if labels is not None:
            axes[i].set_title(str(labels[i]), fontsize=9)
    for j in range(n, len(axes)):
        axes[j].axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


if dat_files and csv_path:
    outlines = load_outlines(csv_path)
    images_by_stem = {os.path.splitext(os.path.basename(p))[0]: p for p in images}
    demo = None
    for dp in dat_files:
        stem = os.path.splitext(os.path.basename(dp))[0]
        if stem in images_by_stem and os.path.basename(images_by_stem[stem]) in outlines:
            demo = (dp, images_by_stem[stem],
                    outlines[os.path.basename(images_by_stem[stem])][1])
            break
    if demo:
        dp, img_path, pts = demo
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pts = pts.astype(int)
        plt.figure(figsize=(7, 5))
        plt.imshow(img)
        for (x, y) in pts:
            plt.plot(x, y, "ro", markersize=6)
        plt.title(os.path.basename(img_path) + " (red dots = ground-truth corners)")
        plt.axis("off")
        plt.show()
        print("ground-truth puzzle (0 = empty cell):")
        print(parse_dat(dp))
    else:
        print("found .dat files and outlines, but no matching image - check the download.")
else:
    print("Dataset not found yet. Use the provisioning section above, or continue "
          "to train on synthetic digits only.")


## 5. Cell preprocessing + extraction of labeled digit crops

This is the preprocessing every cell goes through, **both during training and at
inference time** - the notebook and the repo's `preprocess_cell` are identical:

1. **Gaussian blur** (3x3) to smooth sensor noise.
2. **Adaptive thresholding** (`ADAPTIVE_THRESH_GAUSSIAN_C`, block size 15, constant 7,
   inverted). This is the key step: instead of one global cutoff it computes a
   per-pixel threshold from the local neighborhood, so uneven lighting and shadows
   don't break the image the way a fixed threshold would. A global Otsu pass is
   used ONLY when adaptive is clearly degenerate (washed-out cells) - the two are
   never compared on plausibility.
3. **Strip a 6% border margin** - removes the thinnest grid-line remnants.
4. **Remove grid-line fragments by SHAPE** (the dominant photo failure mode was
   empty cells whose thresholded grid lines survived as the "largest component"
   and were read as digits):
   - lines touching two OPPOSITE borders (edge-to-edge): thin (area/length < 6 px)
     and elongated (bbox aspect > 4) - grid lines run edge to edge, digits don't
   - lines touching any one border: thin (< 2.5 px), elongated, spanning >= 70%
     of the region
   - L-shaped corner remnants: touching two adjacent borders, spanning >= 40% of
     the region in both dims, area < 6%
5. **Drop tiny components** (< 1% of the cell).
6. **Conditionally rejoin split stroke pieces** (e.g. a 7's stem and bar): pieces
   within 5 px, aligned, each >= 1% of the cell, union still digit-like.
7. **Keep only the largest connected component** (specks < 1% -> empty cell).
8. **Crop tightly around the digit's bounding box** with 12% padding.
9. **Resize to 48x48, normalize to float32 [0,1]** and add a channel dimension
   -> shape `(48,48,1)`, exactly what `DigitCNN` / `predict_cells` expect.

All fractions are measured against the FULL cell area, and the cell goes
all-black only when nothing credible survives - thin real strokes are never
rejected by ink heuristics.

Then the grid is perspective-warped from the outline corners into a 600x600 square and
split into 81 row-major cells. **Every** cell is extracted and labeled from the `.dat`
ground truth - digits 1-9 **and** empty cells (class 0). These photo cells ARE the
optional training source for the model (`CONFIG["include_photo"]`): they are pulled
from the SEALED train+val partitions of the Lexski split in section 7. The sealed
TEST partition is never added to any training pool - it is evaluated once, in
section 13 (detect -> recognize -> compare against the ground-truth grid).


In [ ]:
# --- 5. Preprocessing + extraction ---------------------------------------------
def order_points(pts):
    """Reorder 4 corners: top-left, top-right, bottom-right, bottom-left."""
    pts = pts.reshape(4, 2).astype("float32")
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect


def warp_grid(img, pts, size=600):
    """Perspective-correct the grid so it fills a 600x600 square."""
    rect = order_points(pts)
    dst = np.array([[0, 0], [size - 1, 0], [size - 1, size - 1], [0, size - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(img, M, (size, size))


def extract_cells(warped, size=600):
    """Split the warped grid into 81 cells, row-major order (index r*9+c).

    FRACTIONAL boundaries (i*size/9 rounded): size // 9 truncates (600 -> 66px),
    misaligning every later row/column and discarding the last 6px of the warp.
    """
    h, w = warped.shape[:2]
    rows = [int(round(i * h / 9)) for i in range(10)]
    cols = [int(round(i * w / 9)) for i in range(10)]
    cells = []
    for r in range(9):
        for c in range(9):
            cells.append(warped[rows[r]:rows[r + 1], cols[c]:cols[c + 1]])
    return cells


def preprocess_cell(cell, target=48):
    """Phase 1 lenient preprocessing -> (48,48,1) float32.

    Identical algorithm to the repo's digit_cnn.preprocess_cell (v5):
    adaptive threshold (block 15, constant 7, inverted) with an Otsu
    fallback ONLY when adaptive is clearly degenerate (< 0.5% ink AND
    std > 18 - a pure degeneracy gate, never an ink comparison), then:
    10% margin strip, shape-based fragment removal (edge-to-edge lines
    up to 6 px thin + elongated, one-border lines < 2.5 px spanning
    >= 70%, L-shaped corner remnants), tiny-component drop (< 0.5% of
    the FULL cell), split strokes conditionally re-joined with a THIN
    BRIDGE, ALL surviving components kept (no largest-component-only),
    conservative empty (largest surviving < 1% -> all-black),
    letterboxed aspect-preserving resize.
    """
    if cell.ndim == 3:
        cell = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    if cell.dtype != np.uint8:
        cell = np.clip(cell, 0, 255).astype(np.uint8)
    blur = cv2.GaussianBlur(cell, (3, 3), 0)
    th = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 15, 7)
    if (th > 0).mean() < 0.005 and blur.std() > 18:
        th = cv2.threshold(blur, 0, 255,
                           cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    h, w = th.shape
    m = max(1, int(round(min(h, w) * 0.10)))   # margin strip (~7px on a 66px cell)
    th[:m, :] = 0
    th[-m:, :] = 0
    th[:, :m] = 0
    th[:, -m:] = 0
    inner = th[m:h - m, m:w - m]               # component work on the stripped region
    hh, ww = inner.shape
    min_area = h * w * 0.005                   # tiny threshold: 0.5% of the FULL cell
    if (inner > 0).any():
        n, labels, cc, _ = cv2.connectedComponentsWithStats(inner, 8)
        for i in range(1, n):
            x, y, cw, chh, area = cc[i, 0], cc[i, 1], cc[i, 2], cc[i, 3], cc[i, 4]
            if area < min_area:
                inner[labels == i] = 0
                continue
            length = max(cw, chh)
            thin = area / max(length, 1) < 6.0
            elongated = length / max(min(cw, chh), 1) > 4.0
            if ((x == 0 and x + cw == ww) or (y == 0 and y + chh == hh)) and thin and elongated:
                inner[labels == i] = 0                     # edge-to-edge line fragment
                continue
            touch = (x == 0 or x + cw == ww or y == 0 or y + chh == hh)
            thin2 = area / max(length, 1) < 2.5
            if touch and thin2 and elongated and length >= 0.7 * max(hh, ww):
                inner[labels == i] = 0                     # one-sided line remnant
                continue
            if (((x == 0 or x + cw == ww) and (y == 0 or y + chh == hh)) and
                    cw >= 0.4 * ww and chh >= 0.4 * hh and area < 0.06 * h * w):
                inner[labels == i] = 0                     # L-shaped corner remnant
    # conditional merge of split stroke pieces (e.g. a 7's stem and bar):
    # a THIN BRIDGE joins the pair (not a rect fill, which would give a
    # broken stroke the look of a filled blob); only pieces >= 0.5% of the
    # full cell may merge, so noise specks never fuse into a phantom digit
    for _ in range(4):
        n, labels, cc, _ = cv2.connectedComponentsWithStats(inner, 8)
        if n <= 2:
            break
        comps = [(i, cc[i, 0], cc[i, 1], cc[i, 2], cc[i, 3], cc[i, 4]) for i in range(1, n)]
        best = None
        for ai in range(len(comps)):
            for bi in range(ai + 1, len(comps)):
                ia, xa, ya, wa, ha, aa = comps[ai]
                ib, xb, yb, wb, hb, ab = comps[bi]
                if aa < min_area or ab < min_area:
                    continue
                if ya + ha <= yb:
                    gap, ov, ov_ref = yb - (ya + ha), min(xa + wa, xb + wb) - max(xa, xb), min(wa, wb)
                    bx = (max(xa, xb) + min(xa + wa, xb + wb)) // 2
                    (x1, y1), (x2, y2) = (bx, ya + ha), (bx, yb)
                elif yb + hb <= ya:
                    gap, ov, ov_ref = ya - (yb + hb), min(xa + wa, xb + wb) - max(xa, xb), min(wa, wb)
                    bx = (max(xa, xb) + min(xa + wa, xb + wb)) // 2
                    (x1, y1), (x2, y2) = (bx, yb + hb), (bx, ya)
                elif xa + wa <= xb:
                    gap, ov, ov_ref = xb - (xa + wa), min(ya + ha, yb + hb) - max(ya, yb), min(ha, hb)
                    by = (max(ya, yb) + min(ya + ha, yb + hb)) // 2
                    (x1, y1), (x2, y2) = (xa + wa, by), (xb, by)
                elif xb + wb <= xa:
                    gap, ov, ov_ref = xa - (xb + wb), min(ya + ha, yb + hb) - max(ya, yb), min(ha, hb)
                    by = (max(ya, yb) + min(ya + ha, yb + hb)) // 2
                    (x1, y1), (x2, y2) = (xb + wb, by), (xa, by)
                else:
                    continue
                if gap < 0 or gap > 5 or ov < 0.5 * ov_ref:
                    continue
                bx0, bx1 = min(xa, xb), max(xa + wa, xb + wb)
                by0, by1 = min(ya, yb), max(ya + ha, yb + hb)
                aspect = max(bx1 - bx0, by1 - by0) / max(min(bx1 - bx0, by1 - by0), 1)
                if aspect <= 3.5 and aa + ab >= min_area:
                    if best is None or aa + ab > best[0]:
                        best = (aa + ab, ia, ib, x1, y1, x2, y2)
        if best is None:
            break
        _, _, _, x1, y1, x2, y2 = best
        cv2.line(inner, (x1, y1), (x2, y2), 255, 2)
    n, labels, cc, _ = cv2.connectedComponentsWithStats(inner, 8)
    if n > 1:                                # keep ALL surviving components;
        areas = cc[1:, 4]                    # no largest-component-only step
        if areas.max() < h * w * 0.01:       # conservative empty: nothing
            inner[:] = 0                     # credible survived (< 1% full cell)
    ys, xs = np.nonzero(inner)
    if len(xs) == 0:
        return np.zeros((target, target, 1), dtype=np.float32)   # empty cell
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    pad = int(max(x1 - x0, y1 - y0) * 0.12) + 2  # padding around the digit
    crop = inner[max(0, y0 - pad):y1 + pad + 1, max(0, x0 - pad):x1 + pad + 1]
    ch, cw = crop.shape
    scale = target / max(ch, cw)           # aspect-ratio-preserving resize
    nh, nw = max(1, int(round(ch * scale))), max(1, int(round(cw * scale)))
    resized = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target, target), dtype=np.float32)
    y0c, x0c = (target - nh) // 2, (target - nw) // 2
    canvas[y0c:y0c + nh, x0c:x0c + nw] = resized
    return (canvas / 255.0)[..., None]


# --- extract every labeled cell from every photo listed in the outlines csv -
# The CSV is the master list: for every (image, corners) row we look up the
# matching .dat ground truth next to the image and crop the 81 cells. This is
# only used to SHOW what the preprocessing produces (the Lexski photo cells
# in section 7b come from the sealed split instead of this CSV).
X_photo = np.zeros((0, 48, 48, 1), dtype=np.float32)
y_photo = np.zeros((0,), dtype=np.int64)
photo_id = np.zeros((0,), dtype=np.int64)
if csv_path:
    outlines = load_outlines(csv_path)
    crops, labels, phids, used = [], [], [], 0
    for fname, (img_path, pts) in outlines.items():
        dat_path = os.path.join(os.path.dirname(img_path),
                                os.path.splitext(fname)[0] + ".dat")
        if not os.path.exists(dat_path):
            continue
        try:
            grid = parse_dat(dat_path)
            img = cv2.imread(img_path)
            if img is None:
                continue
            warped = warp_grid(img, pts)
            for cell, d in zip(extract_cells(warped), grid.ravel()):
                crops.append(preprocess_cell(cell))
                labels.append(int(d))
                phids.append(used)
            used += 1
        except Exception as exc:
            print("skip", fname, exc)
    if crops:
        X_photo = np.stack(crops).astype(np.float32)
        y_photo = np.array(labels, dtype=np.int64)
        photo_id = np.array(phids, dtype=np.int64)
    print(f"processed {used} photos -> {len(y_photo)} labeled cell crops")
    print("per-class counts:", {k: int((y_photo == k).sum()) for k in range(10)})
    print("empty cells (class 0):", int((y_photo == 0).sum()))
    show_montage(X_photo, y_photo, n=16, seed=SEED, classes=y_photo,
                 title="real-photo cell crops (preprocessed)")
else:
    print("No labeled dataset available - continuing with synthetic digits only.")


## 6. Synthetic empty cells (class 0)

Sudoku puzzles never contain the digit 0, so class 0 means **empty cell**. It is
generated to look like what the CURRENT `preprocess_cell` produces on REAL empty
photo cells (measured on the labeled benchmark photos under the Phase 1
pipeline): ~97% pure black - the preprocessing now strips grid fragments and
tiny specks - and ~3% carrying a surviving thin line fragment or small blob
(5-15% ink). This matters: if class 0 were only all-black images, the CNN would
read any textured photo cell as a digit.
`empty_per_class` (default 15000) matches the per-class digit counts from MNIST +
curated print-font synthetic so the empty class is not starved. Digits 1-9 come
exclusively from MNIST (handwritten) and the curated print-font generator - no
weird decorative fonts, no noisy house-number photos.


In [ ]:
# --- 6. Synthetic empty cells (class 0) ----------------------------------------
# Class 0 = EMPTY cell, NOT the digit 0. The samples are synthesized to match
# the post-preprocessing appearance of REAL empty photo cells (measured on the
# labeled benchmark photos under the Phase 1 pipeline): ~97% pure black, ~3%
# with a surviving thin line fragment / small blob at 5-15% ink. Without this,
# the CNN learns "all black = empty" and reads any textured photo cell as a digit.
# Fragments are drawn at full intensity (thresholded pipeline output is binary
# white), then pushed through `_letterbox_fragment` - the SAME tight-crop/12%
# pad/aspect-preserving-resize geometry the finish step applies to real
# survivors, so the generated empties match the post-cleanup look.
def _letterbox_fragment(img, target):
    ys, xs = np.nonzero(img)
    if len(xs) == 0:
        return img
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    pad = int(max(x1 - x0, y1 - y0) * 0.12) + 2
    crop = img[max(0, y0 - pad):y1 + pad + 1, max(0, x0 - pad):x1 + pad + 1]
    ch, cw = crop.shape
    scale = target / max(ch, cw)
    nh, nw = max(1, int(round(ch * scale))), max(1, int(round(cw * scale)))
    resized = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target, target), dtype=np.float32)
    y0c, x0c = (target - nh) // 2, (target - nw) // 2
    canvas[y0c:y0c + nh, x0c:x0c + nw] = resized
    return canvas


def make_empty_cells(n_per_class=15000, target=48, seed=0):
    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n_per_class):
        img = np.zeros((target, target), dtype=np.float32)
        if rng.random() < 0.97:
            pass                                # clean empty (~97%)
        else:
            kind = np_rng.choice(["blob", "line", "big"], p=[0.5, 0.35, 0.15])
            if kind == "blob":                  # small dirt blob (5-10%)
                cx = np_rng.uniform(14, target - 14)
                cy = np_rng.uniform(14, target - 14)
                rx, ry = np_rng.uniform(4, 7), np_rng.uniform(4, 7)
                cv2.ellipse(img, (int(cx), int(cy)), (int(rx), int(ry)),
                            np_rng.uniform(0, 180), 0, 360, 1.0, -1)
            elif kind == "line":                # thin fragment (3-6%)
                if rng.random() < 0.5:
                    y0 = int(np_rng.uniform(4, target - 6))
                    x0 = int(np_rng.uniform(2, target - 2))
                    img[y0:y0 + int(np_rng.uniform(1, 3)),
                        x0:x0 + int(np_rng.uniform(20, target - 4))] = 1.0
                else:
                    x0 = int(np_rng.uniform(4, target - 6))
                    y0 = int(np_rng.uniform(2, target - 2))
                    img[y0:y0 + int(np_rng.uniform(20, target - 4)),
                        x0:x0 + int(np_rng.uniform(1, 3))] = 1.0
            else:                               # big blob (10-15%)
                cx = np_rng.uniform(16, target - 16)
                cy = np_rng.uniform(16, target - 16)
                rx, ry = np_rng.uniform(7, 10), np_rng.uniform(7, 10)
                cv2.ellipse(img, (int(cx), int(cy)), (int(rx), int(ry)),
                            np_rng.uniform(0, 180), 0, 360, 1.0, -1)
            img = _letterbox_fragment(img, target)
        img = cv2.GaussianBlur(img, (3, 3), 0)  # soften (anti-alias)
        X.append(np.clip(img, 0, 1)[..., None].astype(np.float32))
        y.append(0)
    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)


X_empty, y_empty = make_empty_cells(n_per_class=CONFIG["empty_per_class"], seed=SEED)
print("empty cells (class 0):", X_empty.shape)
show_montage(X_empty, y_empty, n=20, seed=SEED, classes=y_empty,
             title="synthetic empty cells (class 0)")


In [ ]:
# --- 6. (helpers used by augmentation) ----------------------------------------
def rotate(img, angle):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def translate(img, dx, dy):
    # realistic camera movement: pixels pushed out of frame are lost, not
    # wrapped around to the opposite edge (unlike np.roll)
    h, w = img.shape
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def center_crop_or_pad(img, target):
    h, w = img.shape
    if h >= target and w >= target:
        y0, x0 = (h - target) // 2, (w - target) // 2
        return img[y0:y0 + target, x0:x0 + target]
    canvas = np.zeros((target, target), dtype=img.dtype)
    y0, x0 = (target - h) // 2, (target - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = img
    return canvas


## 6b. MNIST handwritten-digit supplement (optional)

Real-world sudoku photos are often partially solved by hand - someone has already
filled cells with a pen - or the whole puzzle is hand-drawn. To make the classifier
robust to handwriting too, we add the classic MNIST handwritten-digit corpus. MNIST is
28x28, so the samples are upscaled and passed through the same `preprocess_cell`.
**MNIST digit-0 samples are dropped** (class 0 means "empty cell" in this model, not
the digit zero). Set `CONFIG["include_mnist"] = False` to skip this.


In [ ]:
# --- 6b. MNIST handwritten digits (optional) ----------------------------------
def load_mnist(root="/content/mnist", target=48, n_per_class=0, seed=0):
    from torchvision import datasets
    from torchvision.transforms import ToTensor

    def cap(X, y):
        if n_per_class:
            rng = np.random.default_rng(seed)
            keep = []
            for k in np.unique(y):
                idx = np.where(y == k)[0]
                rng.shuffle(idx)
                keep.append(idx[:n_per_class])
            sel = np.concatenate(keep)
            return X[sel], y[sel]
        return X, y

    def pack(ds):
        X, y = [], []
        for img, label in ds:
            if int(label) == 0:
                continue            # class 0 means EMPTY cell, not the digit 0
            arr = img.numpy()[0]                       # 28x28 float32, WHITE on black
            arr = 1.0 - arr                            # invert: dark ink on paper
            arr = cv2.resize(arr, (56, 56), interpolation=cv2.INTER_CUBIC)
            arr = np.clip(arr, 0.0, 1.0)               # cubic resize can overshoot
            # run the SAME preprocess_cell used for real photo cells
            X.append(preprocess_cell((arr * 255.0).astype(np.uint8)))
            y.append(int(label))
        return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)

    train = datasets.MNIST(root=root, train=True, download=True, transform=ToTensor())
    test = datasets.MNIST(root=root, train=False, download=True, transform=ToTensor())
    X, y = cap(*pack(train))
    Xt, yt = cap(*pack(test))
    # official TEST split kept separate - never concatenated into the pool
    return X, y, Xt, yt


X_mnist = np.zeros((0, 48, 48, 1), dtype=np.float32)
y_mnist = np.zeros((0,), dtype=np.int64)
X_mnist_te = np.zeros((0, 48, 48, 1), dtype=np.float32)
y_mnist_te = np.zeros((0,), dtype=np.int64)
if CONFIG.get("include_mnist"):
    X_mnist, y_mnist, X_mnist_te, y_mnist_te = load_mnist(
        root=CONFIG["mnist_root"],
        n_per_class=CONFIG.get("mnist_per_class", 0), seed=SEED)
    print("mnist:", X_mnist.shape, "| per-class:", np.bincount(y_mnist).tolist())
    show_montage(X_mnist, y_mnist, n=16, seed=SEED, classes=y_mnist,
                 title="MNIST handwritten digits (through cell preprocessing)")
else:
    print("include_mnist=False - skipping MNIST")


## 6c. EMNIST handwritten digits (optional)

EMNIST's `digits` split is a ~10x larger, more varied handwriting corpus in the
same style as MNIST (280k samples). Same treatment: upscaled 28->48 and passed
through the same `preprocess_cell`; **digit-0 samples are dropped** (class 0 means
"empty cell"). Set `CONFIG["include_emnist"] = False` to skip, or cap with
`CONFIG["emnist_per_class"]` (0 = use everything).


In [ ]:
# --- 6c. EMNIST handwritten digits (optional) ---------------------------------
def load_emnist(root="/content/emnist", target=48, n_per_class=0, seed=0):
    from torchvision import datasets
    from torchvision.transforms import ToTensor

    cache_path = os.path.join(root, "emnist_packed.npz")
    te_cap = (n_per_class // 4) if n_per_class else 0
    if os.path.exists(cache_path):
        data = np.load(cache_path)
        if (data.get("preprocess_version") is not None
                and str(data["preprocess_version"].item()) == "v5"
                and all(k in data for k in ("X_tr", "y_tr", "X_te", "y_te"))):
            X = data["X_tr"].astype(np.float32)
            y = data["y_tr"].astype(np.int64)
            Xt = data["X_te"].astype(np.float32)
            yt = data["y_te"].astype(np.int64)
            return _cap_emnist(X, y, n_per_class, seed),                    _cap_emnist(Xt, yt, te_cap, seed)

    def pack(ds):
        X, y = [], []
        for img, label in ds:
            if int(label) == 0:
                continue            # class 0 means EMPTY cell, not the digit 0
            arr = img.numpy()[0]                       # 28x28, WHITE on black
            arr = 1.0 - arr                            # invert: dark ink on paper
            arr = cv2.resize(arr, (56, 56), interpolation=cv2.INTER_CUBIC)
            arr = np.clip(arr, 0.0, 1.0)               # cubic resize can overshoot
            X.append(preprocess_cell((arr * 255.0).astype(np.uint8)))
            y.append(int(label))
        return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)

    train = datasets.EMNIST(root=root, split="digits", train=True,
                            download=True, transform=ToTensor())
    test = datasets.EMNIST(root=root, split="digits", train=False,
                           download=True, transform=ToTensor())
    Xt, yt = pack(train)
    Xe, ye = pack(test)
    os.makedirs(root, exist_ok=True)
    np.savez(cache_path, X_tr=Xt, y_tr=yt, X_te=Xe, y_te=ye,
             preprocess_version="v5")
    return _cap_emnist(Xt, yt, n_per_class, seed),            _cap_emnist(Xe, ye, te_cap, seed)


def _cap_emnist(X, y, n_per_class, seed):
    if n_per_class:
        rng = np.random.default_rng(seed)
        keep = []
        for k in np.unique(y):
            idx = np.where(y == k)[0]
            rng.shuffle(idx)
            keep.append(idx[:n_per_class])
        sel = np.concatenate(keep)
        X, y = X[sel], y[sel]
    return X, y


X_emnist = np.zeros((0, 48, 48, 1), dtype=np.float32)
y_emnist = np.zeros((0,), dtype=np.int64)
X_emnist_te = np.zeros((0, 48, 48, 1), dtype=np.float32)
y_emnist_te = np.zeros((0,), dtype=np.int64)
if CONFIG.get("include_emnist"):
    try:
        (X_emnist, y_emnist), (X_emnist_te, y_emnist_te) = load_emnist(
            root=CONFIG["emnist_root"],
            n_per_class=CONFIG.get("emnist_per_class", 0),
            seed=SEED)
        print("emnist:", X_emnist.shape, "| per-class:", np.bincount(y_emnist).tolist())
        show_montage(X_emnist, y_emnist, n=16, seed=SEED, classes=y_emnist,
                     title="EMNIST handwritten digits (through cell preprocessing)")
    except Exception as exc:
        print(f"EMNIST unavailable ({exc}); continuing without it.")
else:
    print("include_emnist=False - skipping EMNIST")


## 6d. Curated print-font synthetic digits (optional)

SVHN is real-world but noisy; sudoku puzzles use uniform printed digits. This
generator renders digits with **common print fonts only** (DejaVu, Arial, Times,
Courier, Verdana, Georgia, Tahoma, Corbel, Segoe UI, Liberation, Free*) - no
decorative/handwriting fonts, which produced the weird samples before. Every
render is passed through the same `preprocess_cell` as everything else.
Unlimited clean printed digits that train to ~99%. Set `CONFIG["include_synth"] = False`
to skip.


In [ ]:
# --- 6d. Curated print-font synthetic digits (optional) -----------------------
def find_print_fonts():
    wanted = ("dejavu", "arial", "times", "cour", "verdana", "georgia", "tahoma",
              "corbel", "segoeui", "liberation", "freesans", "freeserif")
    dirs = []
    if os.name == "nt":
        dirs.append(r"C:\Windows\Fonts")
    for d in ["/usr/share/fonts/truetype/dejavu", "/usr/share/fonts/truetype/liberation",
              "/usr/share/fonts/truetype/freefont", "/usr/share/fonts"]:
        if os.path.isdir(d):
            dirs.append(d)
    found = []
    for d in dirs:
        for name in os.listdir(d):
            low = name.lower()
            if low.endswith((".ttf", ".ttc")) and any(k in low for k in wanted):
                found.append(os.path.join(d, name))
    return sorted(set(found))


def render_digit(d, font_path, size=64):
    img = Image.new("L", (size, size), 0)
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(font_path, int(size * 0.8))
    bbox = draw.textbbox((0, 0), d, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (size - w) // 2 - bbox[0]
    y = (size - h) // 2 - bbox[1]
    draw.text((x, y), d, fill=255, font=font)
    return np.asarray(img, dtype=np.float32)


def make_synthetic_digits(n_per_class=4000, target=48, seed=0, sizes=(52, 56, 64, 72)):
    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)
    fonts = find_print_fonts()
    if not fonts:
        print("no print fonts found - skipping synthetic digits")
        return np.zeros((0, 48, 48, 1), dtype=np.float32), np.zeros((0,), dtype=np.int64)
    X, y = [], []
    for k in range(1, 10):
        for _ in range(n_per_class):
            font = rng.choice(fonts)
            size = rng.choice(sizes)
            img = render_digit(str(k), font, size)     # white on black
            # --- augment the RAW cell BEFORE preprocessing (conservative) ----
            scale = np_rng.uniform(0.8, 1.0)           # small zoom in/out
            img = cv2.resize(img, (int(target / scale), int(target / scale)),
                             interpolation=cv2.INTER_AREA)
            img = center_crop_or_pad(img, target)
            dx, dy = int(np_rng.uniform(-2, 3)), int(np_rng.uniform(-2, 3))
            img = translate(img, dx, dy)               # shift +/-2 px (black border)
            if rng.random() < 0.8:                     # camera tilt (mild)
                img = rotate(img, np_rng.uniform(-8, 8))
                img = center_crop_or_pad(img, target)
            if rng.random() < 0.4:                     # print weight: thin/heavy
                ker = np.ones((3, 3), np.uint8)
                if rng.random() < 0.5:
                    img = cv2.erode(img, ker, iterations=1)
                else:
                    img = cv2.dilate(img, ker, iterations=1)
            if rng.random() < 0.6:                     # lighting: brightness/contrast
                gain = np_rng.uniform(0.75, 1.25)
                bias = np_rng.uniform(-25, 25)
                img = np.clip(img * gain + bias, 0, 255)
            if rng.random() < 0.6:                     # weak sensor/paper noise
                img = img + np_rng.normal(0, np_rng.uniform(2, 6), img.shape)
            if rng.random() < 0.2:                     # occasional light blur
                img = cv2.GaussianBlur(img, (3, 3), 0)
            if rng.random() < 0.05:                    # rare strong blur
                img = cv2.GaussianBlur(img, (5, 5), 0)
            raw = np.clip(255 - img, 0, 255).astype(np.uint8)   # dark on light
            X.append(preprocess_cell(raw))
            y.append(k)
    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)


X_synth = np.zeros((0, 48, 48, 1), dtype=np.float32)
y_synth = np.zeros((0,), dtype=np.int64)
if CONFIG.get("include_synth"):
    X_synth, y_synth = make_synthetic_digits(n_per_class=CONFIG["synth_per_class"], seed=SEED)
    print("synth:", X_synth.shape, "| per-class:", np.bincount(y_synth).tolist())
    show_montage(X_synth, y_synth, n=18, seed=SEED, classes=y_synth,
                 title="curated print-font synthetic digits (through preprocessing)")
else:
    print("include_synth=False - skipping synthetic digits")


## 7. Combine sources and split 70 / 15 / 15 (best practice)

Training is done on **controlled digit sources** (synthetic empty cells (class 0),
curated print-font synthetic digits and MNIST/EMNIST handwriting, all 1-9) pooled
with a `source` label per sample and split **stratified by digit class** with a
fixed seed. When `CONFIG["include_photo"]` is set, **real photo cells** from the
Lexski train+val partitions join the pool; their sealed partition membership is
kept (photo train -> train, photo val -> val) - the photo TEST partition is never
loaded here.

| split | share | how it is used |
|-------|-------|----------------|
| train | 70%   | gradient updates + augmentation (incl. photo train partition) |
| val   | 15%   | picking the best checkpoint only (incl. photo val partition) |
| test  | 15%   | **held out completely** - evaluated exactly once, at the end |

Stratification guarantees every digit class appears in every split in the same
proportion; the fixed seed makes the partition reproducible. The `source` labels let us
report accuracy per source (empty / synth / mnist / emnist / photo) at the end, so we
can see how each domain contributes.


In [ ]:
# --- 7b. Real photo cells (Lexski, SEALED 70/15/15 split) ---------------------
# Rebuilds the SAME sealed split as the repo's photo_data.py (seed 42, sorted
# names, 70/15/15 over all 1400 images) from the three metadata.jsonl files,
# then extracts definite cells from the TRAIN+VAL partitions exactly like the
# repo's recognition benchmark path: GT corners -> warp_grid -> fractional
# extract_cells -> preprocess_cell. The TEST partition is sealed - its images
# are never downloaded here (section 13 benchmarks it).
if CONFIG.get("include_photo"):
    import urllib.request as _urlreq
    from concurrent.futures import ThreadPoolExecutor

    hf_base = ("https://huggingface.co/datasets/Lexski/sudoku-image-recognition/"
               "resolve/main/data")
    hf_dir = CONFIG["photo_hf_dir"]
    hf_splits = ("train", "val", "test")

    def hf_meta_rows(split):
        d = os.path.join(hf_dir, split)
        os.makedirs(d, exist_ok=True)
        p = os.path.join(d, "metadata.jsonl")
        if not os.path.exists(p):
            _urlreq.urlretrieve(f"{hf_base}/{split}/metadata.jsonl", p)
        return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

    def hf_cells_to_gt_local(cells):
        """(9,9,10) flags -> (gt_grid, definite_mask) - local copy; the
        benchmark section redefines the same helper later (cell order)."""
        cells = np.asarray(cells).reshape(9, 9, 10)
        gt = np.zeros((9, 9), dtype=int)
        definite = np.zeros((9, 9), dtype=bool)
        for r in range(9):
            for c in range(9):
                flags = cells[r, c]
                solved = int(flags[0]) == 1
                digits = [d for d in range(1, 10) if int(flags[d]) == 1]
                if solved and len(digits) == 1:
                    gt[r, c] = digits[0]
                    definite[r, c] = True
                elif not solved and len(digits) == 0:
                    definite[r, c] = True
        return gt, definite

    meta = {s: hf_meta_rows(s) for s in hf_splits}
    name_to_split = {}
    for s, rows in meta.items():
        for row in rows:
            name_to_split[os.path.basename(
                row["file_name"].replace("\\", "/"))] = s
    names = sorted(name_to_split)
    rng = np.random.RandomState(CONFIG["photo_split_seed"])
    perm = rng.permutation(len(names))
    n_tr = int(round(len(names) * 0.70))
    n_va = int(round(len(names) * 0.15))
    parts = {"train": set(names[i] for i in perm[:n_tr]),
             "val": set(names[i] for i in perm[n_tr:n_tr + n_va]),
             "test": set(names[i] for i in perm[n_tr + n_va:])}
    print("sealed split (seed 42):",
          {k: len(v) for k, v in parts.items()})

    jobs = []
    for part in ("train", "val"):
        for name in parts[part]:
            s = name_to_split[name]
            jobs.append((s, name, f"{hf_base}/{s}/images/{name}"))

    def fetch(split, name, url):
        out = os.path.join(hf_dir, split, "images", name)
        if not os.path.exists(out):
            os.makedirs(os.path.dirname(out), exist_ok=True)
            _urlreq.urlretrieve(url, out)

    with ThreadPoolExecutor(max_workers=8) as ex:
        for _ in ex.map(lambda j: fetch(*j), jobs):
            pass
    print(f"downloaded {len(jobs)} photo images (train+val partitions, "
          f"test sealed)")

    def photo_pool(part):
        X, y = [], []
        for name in sorted(parts[part]):
            s = name_to_split[name]
            img = cv2.imread(os.path.join(hf_dir, s, "images", name))
            if img is None:
                continue
            row = next(r for r in meta[s] if os.path.basename(
                r["file_name"].replace("\\", "/")) == name)
            kp = np.array(row["keypoints"], dtype=np.float32).reshape(4, 2)
            kp = kp[[0, 3, 2, 1]]                  # TL,BL,BR,TR -> TL,TR,BR,BL
            gt, definite = hf_cells_to_gt_local(row["cells"])
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            warped = warp_grid(gray, kp)
            for i, cell in enumerate(extract_cells(warped)):
                r, c = divmod(i, 9)
                if definite[r, c]:
                    X.append(preprocess_cell(cell))
                    y.append(int(gt[r, c]))
        return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)

    X_photo_tr, y_photo_tr = photo_pool("train")
    X_photo_va, y_photo_va = photo_pool("val")
    print(f"photo cells: train {len(y_photo_tr)} | val {len(y_photo_va)}")
else:
    X_photo_tr = X_photo_va = np.zeros((0, 48, 48, 1), dtype=np.float32)
    y_photo_tr = y_photo_va = np.zeros((0,), dtype=np.int64)


In [ ]:
# --- 7. Combine + split ---------------------------------------------------------
# Controlled sources: synthetic empty cells (class 0), curated print-font
# synthetic digits and MNIST/EMNIST handwriting (both 1-9). Photo cells from
# the sealed Lexski TRAIN+VAL partitions (cell 7b) join with their partition
# membership kept; the sealed photo TEST partition is never loaded here.
#
# MEMORY: the naive `X = np.concatenate([...])` + `X[tr_idx]` held every source
# array alive while materializing full-size copies (peak ~12 GB with uncapped
# EMNIST on a 12.7 GB Colab session) and killed the kernel. The splits are
# instead gathered PER SOURCE below: each source contributes one transient
# slice per split and is freed immediately, so the peak stays ~7 GB. Split
# MEMBERSHIP is identical to the naive version; intra-split order is grouped
# by source (every consumer uses random sampling or masks, so order does not
# matter).
SOURCES = [("empty", X_empty, y_empty), ("synth", X_synth, y_synth),
           ("mnist", X_mnist, y_mnist), ("emnist", X_emnist, y_emnist)]
_OFF = np.concatenate([[0], np.cumsum([len(Xk) for _, Xk, _ in SOURCES])])
phid = np.concatenate([-(np.arange(len(X_empty)) + 1),
                       -(np.arange(len(X_synth)) + 1 + len(X_empty)),
                       -(np.arange(len(X_mnist)) + 1 + len(X_empty) + len(X_synth)),
                       -(np.arange(len(X_emnist)) + 1 + len(X_empty) + len(X_synth)
                         + len(X_mnist))])
print("total samples:", len(phid),
      "| empty:", len(X_empty),
      "| synth:", len(X_synth),
      "| mnist:", len(X_mnist),
      "| emnist:", len(X_emnist))


def group_wise_split(phid, train_frac=0.7, val_frac=0.15, seed=0):
    """Split whole groups (photos) so cells of one photo never straddle splits.

    Groups are shuffled, then each group is assigned to the split that is
    currently furthest below its target share (greedy balancing).
    """
    rng = np.random.RandomState(seed)
    groups = {}
    for i, g in enumerate(phid):
        groups.setdefault(g, []).append(i)
    gkeys = list(groups)
    rng.shuffle(gkeys)
    n = len(phid)
    targets = [int(round(n * train_frac)),
               int(round(n * val_frac)),
               n - int(round(n * train_frac)) - int(round(n * val_frac))]
    counts = [0, 0, 0]
    buckets = [[], [], []]
    for g in gkeys:
        gs = groups[g]
        j = min(range(3), key=lambda j: counts[j] / max(targets[j], 1))
        buckets[j].extend(gs)
        counts[j] += len(gs)
    return np.array(buckets[0]), np.array(buckets[1]), np.array(buckets[2])


tr_idx, va_idx, te_idx = group_wise_split(phid, CONFIG["train_frac"], CONFIG["val_frac"], seed=SEED)


def _assert_no_leakage(phid, tr_idx, va_idx, te_idx):
    """Guard: no group (photo) may appear in more than one split."""
    sets = [set(phid[tr_idx]), set(phid[va_idx]), set(phid[te_idx])]
    for a in range(3):
        for b in range(a + 1, 3):
            common = sets[a] & sets[b]
            if common:
                raise AssertionError(
                    "leakage between splits: groups " + str(sorted(common)[:5]))


_assert_no_leakage(phid, tr_idx, va_idx, te_idx)

# photo cells join with their SEALED partition membership (train->train,
# val->val); the test partition stays unloaded. Their group ids are not in
# `phid`, so the leakage guard above cannot see them - the sealed split file
# construction (cell 7b) already guarantees disjointness.
def _gather_splits(tr_idx, va_idx, te_idx):
    """Per-source gathering -> (X, y, src) tuples for tr/va/te.

    Each source array is fancy-indexed ONCE per split (a transient copy,
    freed before the next source is processed), so the combined `X` and the
    old `X[tr_idx]` full-size copies are never materialized. Sample
    MEMBERSHIP per split is identical to combined-space indexing.
    """
    g = globals()
    out = {"tr": ([], [], []), "va": ([], [], []), "te": ([], [], [])}
    for j, (name, Xk, yk) in enumerate(SOURCES):
        lo, hi = _OFF[j], _OFF[j + 1]
        for key, idx in (("tr", tr_idx), ("va", va_idx), ("te", te_idx)):
            m = (idx >= lo) & (idx < hi)
            if m.any():
                local = idx[m] - lo
                out[key][0].append(Xk[local])
                out[key][1].append(yk[local])
                out[key][2].append(np.full(int(m.sum()), name))
        # free this source's arrays NOW (its slices are already copied out)
        SOURCES[j] = (name, None, None)
        del Xk, yk
        g["X_" + name] = None
        g["y_" + name] = None
    splits = {}
    for key in ("tr", "va", "te"):
        xs, ys, ss = out[key]
        splits[key] = (np.concatenate(xs), np.concatenate(ys), np.concatenate(ss))
    return splits["tr"], splits["va"], splits["te"]


tr_g, va_g, te_g = _gather_splits(tr_idx, va_idx, te_idx)
del SOURCES, _OFF
X_tr = np.concatenate([tr_g[0], X_photo_tr]).astype(np.float32)
y_tr = np.concatenate([tr_g[1], y_photo_tr]).astype(np.int64)
src_tr = np.concatenate([tr_g[2], np.array(["photo"] * len(y_photo_tr))])
del tr_g, X_photo_tr, y_photo_tr
X_va = np.concatenate([va_g[0], X_photo_va]).astype(np.float32)
y_va = np.concatenate([va_g[1], y_photo_va]).astype(np.int64)
src_va = np.concatenate([va_g[2], np.array(["photo"] * len(y_photo_va))])
del va_g, X_photo_va, y_photo_va
X_te, y_te, src_te = te_g
del te_g
print("split sizes -> train:", len(y_tr), "| val:", len(y_va), "| test:", len(y_te))
for s in ("empty", "synth", "mnist", "emnist", "photo"):
    print(f"  {s:9s}: train {(src_tr == s).sum():6d} | val {(src_va == s).sum():5d} | "
          f"test {(src_te == s).sum():5d}")
print("no leakage: every group stays in one split (asserted)")
print("sealed photo test partition: never loaded during training")

# free everything the splits no longer reference (the old cleanup lived in
# the augmentation cell and ran too late - cell 23 used to peak at ~12 GB)
del phid, tr_idx, va_idx, te_idx
import gc
gc.collect()
show_montage(X_te, y_te, n=16, seed=SEED, classes=y_te,
             title="held-out test samples (untouched until final evaluation)")


## 8. Data augmentation (train split only)

The validation and test splits are left **completely untouched** so the final numbers
describe unmodified data. Only the train slice gets the same stochastic distortions the
synthetic generator uses (small rotations, shifts, Gaussian noise, blur) - effectively
expanding the training set.

Augmentation is applied **lazily, per epoch** (a fresh random transform per sample per
epoch, via the training loop's DataLoader) - the old eager `np.stack` of ~300k augmented
copies peaked at ~12 GB and killed the Colab kernel. The train loader is also
**source-balanced** (WeightedRandomSampler): empty/synth/mnist/emnist/photo each
contribute equally per epoch, so EMNIST's ~67% share does not dominate the gradient.


In [ ]:
# --- 8. Augment the train split only -------------------------------------------
def augment_cell(img, np_rng):
    """img: (48,48,1) float32 in [0,1] -> conservative random distortion
    (real puzzle cells are fronto-parallel after the grid warp)."""
    im = img[..., 0].copy()
    if np_rng.random() < 0.7:
        im = rotate(im, np_rng.uniform(-8, 8))
        im = center_crop_or_pad(im, 48)
    if np_rng.random() < 0.4:                     # small scale variation
        s = np_rng.uniform(0.9, 1.0)
        im = cv2.resize(im, (int(48 * s), int(48 * s)), interpolation=cv2.INTER_AREA)
        im = center_crop_or_pad(im, 48)
    dx, dy = int(np_rng.uniform(-2, 3)), int(np_rng.uniform(-2, 3))
    im = translate(im, dx, dy)
    if np_rng.random() < 0.6:                     # lighting: brightness/contrast
        gain = np_rng.uniform(0.75, 1.25)
        bias = np_rng.uniform(-0.08, 0.08)
        im = np.clip(im * gain + bias, 0.0, 1.0)
    if np_rng.random() < 0.25:                    # weak sensor noise
        im = im + np_rng.normal(0, np_rng.uniform(0.004, 0.012), im.shape).astype(np.float32)
    if np_rng.random() < 0.2:                     # occasional light blur
        im = cv2.GaussianBlur(im, (3, 3), 0)
    return np.clip(im, 0, 1).astype(np.float32)[..., None]


# LAZY augmentation + source-balanced sampling (mirrors train_local.py):
# X_tr stays RAW; augment_cell runs per-epoch inside the training loop. The
# old eager `np.stack([augment_cell(x, rng) for x in X_tr])` over ~300k
# samples peaked at ~12 GB and killed the Colab kernel - this path is
# O(batch). The WeightedRandomSampler makes each source contribute equally
# per epoch (EMNIST's ~67% share no longer dominates the gradient).
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def build_augmented_loader(X, y, src, batch_size, seed=0):
    class AugmentedDataset(Dataset):
        def __init__(self, X_, y_):
            self.X, self.y = X_, y_
            self.rng = np.random.default_rng(seed)

        def __len__(self):
            return len(self.y)

        def __getitem__(self, i):
            return augment_cell(self.X[i], self.rng), int(self.y[i])

    src_counts = {s: int((src == s).sum()) for s in np.unique(src)}
    weights = np.array([1.0 / src_counts[s] for s in src], dtype=np.float64)
    sampler = WeightedRandomSampler(weights, num_samples=len(X), replacement=True)
    return DataLoader(AugmentedDataset(X, y), batch_size=batch_size, sampler=sampler)


# The per-source arrays and the combined copy were already freed at the end
# of the combine cell (the old cleanup here ran too late - cell 23 used to
# peak at ~12 GB and kill the Colab kernel).
print("train split kept raw (lazy per-epoch augmentation):", X_tr.shape,
      "| val/test untouched:", X_va.shape, X_te.shape)


## 9. DigitCNN

The exact model from the repo: **double-convolution** stages (2x Conv3x3-BN-ReLU
per stage, MaxPool 2) 32 -> 64 -> 128, then a **global-average-pooling head**
(`AdaptiveAvgPool2d(1,1) -> Flatten -> Dropout(0.3) -> Linear(128, 10)`). GAP
replaces the old size-locked `Linear(128*6*6, 10)` flatten head: no positional
bias (real digits drift inside their cells) and ~46k -> ~1.3k head params.


In [ ]:
# --- 9. DigitCNN model -----------------------------------------------------------
class DigitCNN(nn.Module):
    """Double-convolution CNN: 32 -> 64 -> 128 (2x Conv3x3-BN-ReLU per stage,
    MaxPool 2 per stage) with a GLOBAL-AVERAGE-POOLING classifier head
    (AdaptiveAvgPool2d -> Flatten -> Dropout -> Linear(128, 10))."""

    def __init__(self, num_classes=10):
        super().__init__()

        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            block(1, 32),
            block(32, 64),
            block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = DigitCNN().to(device)
print(f"DigitCNN with {sum(p.numel() for p in model.parameters()):,} parameters")


## 10. Training with early stopping + checkpointing

`train_digit_model` trains until the validation score **stops improving**:

* the **score is the mean of per-source accuracies** (empty / synth / mnist /
  emnist / photo), not the aggregate - so the harder handwriting and photo
  domains count as much as the easy synthetic data when picking checkpoints,
* if the score has not improved for `CONFIG["early_stop_patience"]` epochs
  (default 5), training stops and the best model (already saved) is used - no point
  burning GPU time once the model has converged,
* `CONFIG["min_epochs"]` (default 12) guarantees we never stop too early - in
  particular we always train past the first `ReduceLROnPlateau` LR drops, which
  often bring a second wave of improvements,
* `CONFIG["epochs"]` (default 40) is the hard cap in case it keeps improving.

**AdamW** (lr 3e-4, weight decay 1e-4) + **ReduceLROnPlateau** (factor 0.5,
patience 4, monitored on the validation score). The loop **checkpoints after every epoch**:

* `checkpoints/ckpt_epochNN.pt` - full checkpoint: model + optimizer state, epoch,
  best score so far, and the loss/accuracy history. Used to resume.
* `checkpoints/best_model.pt` - copy of the epoch with the highest validation score.

The function **returns the best checkpoint's weights** (it reloads `best_model.pt`
before returning), and the export cell reloads `best_model.pt` again - so
`digit_cnn.pth` is always the best model, never the last epoch's.

A checkpoint can be loaded any time and training continues exactly where it stopped.


In [ ]:
# --- 10. Training loop with checkpointing --------------------------------------
def train_digit_model(X_tr, y_tr, src_tr, X_va, y_va, src_va, epochs=40, batch_size=256,
                      lr=3e-4, device="cpu", ckpt_dir="checkpoints", resume_path=None,
                      seed=0, early_stop_patience=5, min_epochs=12,
                      warm_start_path=None):
    torch.manual_seed(seed)
    device = torch.device(device)
    os.makedirs(ckpt_dir, exist_ok=True)

    model = DigitCNN().to(device)
    if warm_start_path is not None and os.path.exists(warm_start_path):
        model.load_state_dict(torch.load(warm_start_path, map_location=device))
        print(f"warm-started from {warm_start_path} (fine-tune, lr={lr})")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    start_epoch, best_score = 0, -1.0
    hist = {"train_loss": [], "val_loss": [], "val_acc": [], "val_score": [],
            "val_photo_acc": []}

    if resume_path is not None and os.path.exists(resume_path):
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        opt.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"]
        best_score = ckpt.get("best_score", -1.0)
        hist = ckpt.get("histories", hist)
        print(f"resumed from {resume_path} at epoch {start_epoch}")

    # adaptive LR: halve when the validation score has not improved for 4 epochs
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", factor=0.5, patience=4, min_lr=1e-5)

    sources = sorted(set(src_va))
    epochs_no_improve = 0
    for ep in range(start_epoch, epochs):
        model.train()
        tot, cor, loss_sum = 0, 0, 0.0
        # fresh lazy augmentation + fresh sampler order EVERY epoch
        tr_loader = build_augmented_loader(X_tr, y_tr, src_tr,
                                           batch_size, seed=seed + ep)
        for x, yb in tr_loader:
            x = x.permute(0, 3, 1, 2).to(device)
            yb = yb.to(device)
            out = model(x)
            loss = loss_fn(out, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            loss_sum += loss.item() * len(x)
            tot += len(x)
            cor += (out.argmax(1) == yb).sum().item()
        tl, ta = loss_sum / tot, cor / tot

        model.eval()
        with torch.no_grad():
            # batched validation - the full val set in one forward pass would
            # blow up GPU memory on a T4 (activations ~5 GiB at batch 20k)
            vl_sum, n = 0.0, 0
            src_cor = {s: 0 for s in sources}
            src_n = {s: 0 for s in sources}
            for i in range(0, len(X_va), batch_size):
                xb = torch.from_numpy(X_va[i:i + batch_size]).permute(0, 3, 1, 2).to(device)
                yb = torch.from_numpy(y_va[i:i + batch_size]).to(device)
                out = model(xb)
                vl_sum += loss_fn(out, yb).item() * len(xb)
                preds = out.argmax(1)
                for j in range(len(preds)):
                    s = src_va[i + j]
                    src_cor[s] += int(preds[j] == yb[j])
                    src_n[s] += 1
                n += len(xb)
            vl = vl_sum / n
            va_agg = sum(src_cor.values()) / n
            src_acc = {s: src_cor[s] / max(src_n[s], 1) for s in sources}
            # selection metric: MEAN OF PER-SOURCE ACCURACIES, so the hard
            # photo domain counts as much as easy synthetic/MNIST data
            va = float(np.mean([src_acc[s] for s in sources]))

        sched.step(va)

        hist["train_loss"].append(tl)
        hist["val_loss"].append(vl)
        hist["val_acc"].append(va_agg)
        hist["val_score"].append(va)
        hist["val_photo_acc"].append(src_acc.get("photo", float("nan")))
        print(f"epoch {ep + 1}/{epochs}  train loss {tl:.4f} acc {ta:.4f} | "
              f"val loss {vl:.4f} agg acc {va_agg:.4f} | score {va:.4f} | "
              + " ".join(f"{s}={src_acc[s]:.3f}" for s in sources))

        ckpt = {"epoch": ep + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "best_score": max(best_score, va),
                "histories": hist}
        torch.save(ckpt, os.path.join(ckpt_dir, f"ckpt_epoch{ep + 1:02d}.pt"))
        if va > best_score:
            best_score = va
            epochs_no_improve = 0
            torch.save(ckpt, os.path.join(ckpt_dir, "best_model.pt"))
            print(f"  -> new best validation score {va:.4f} (saved best_model.pt)")
        else:
            epochs_no_improve += 1
            print(f"  no improvement ({epochs_no_improve}/{early_stop_patience})")
            if ep + 1 >= min_epochs and epochs_no_improve >= early_stop_patience:
                print(f"early stopping at epoch {ep + 1}: score has not improved "
                      f"for {early_stop_patience} epochs (best {best_score:.4f})")
                break

    # return the BEST model (weights from best_model.pt), not the last epoch's
    bp = os.path.join(ckpt_dir, "best_model.pt")
    if os.path.exists(bp):
        model.load_state_dict(torch.load(bp, map_location=device)["model_state_dict"])
        print(f"loaded best_model.pt (best validation score {best_score:.4f})")
    model.eval()
    return model, hist


In [ ]:
# --- 10b. Run training ----------------------------------------------------------
# With MNIST + EMNIST included each epoch is ~2-4 minutes on a Colab T4 GPU
# (batch 256); early stopping usually ends training between ~12-25 epochs
# (it stops when val accuracy stalls). Much slower on CPU, so use
# Runtime > Change runtime type > T4 GPU.
#
# Checkpoints (and the exported digit_cnn.pth later) are written to Google
# Drive (folder CPECOG1) so a runtime reset is a resume, not a restart.
# The Drive mount prompts for authorization the first time - click through,
# then this cell continues.
DRIVE_DIR = CONFIG["drive_dir"]
if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(os.path.join(DRIVE_DIR, "checkpoints"), exist_ok=True)
CONFIG["ckpt_dir"] = os.path.join(DRIVE_DIR, "checkpoints")
print("checkpoints will be saved to", CONFIG["ckpt_dir"])

RESUME_PATH = None   # set by the resume cell below to continue an interrupted run
model, hist = train_digit_model(X_tr, y_tr, src_tr, X_va, y_va, src_va,
                                epochs=CONFIG["epochs"],
                                batch_size=CONFIG["batch_size"],
                                lr=CONFIG["lr"],
                                device=device,
                                ckpt_dir=CONFIG["ckpt_dir"],
                                resume_path=RESUME_PATH,
                                seed=SEED,
                                early_stop_patience=CONFIG["early_stop_patience"],
                                min_epochs=CONFIG["min_epochs"],
                                warm_start_path=CONFIG.get("warm_start"))

# Temperature scaling (Guo et al. 2017): one scalar T minimizing the NLL of
# softmax(logits/T) on the validation split - on the PHOTO val cells when
# present (the distribution the solver's confidence gate actually faces),
# otherwise on the whole val split. T flattens overconfident photo softmaxes
# so wrong cells stop clearing conf_thresh=0.9 and the solver's correction
# search actually revisits them.
def fit_temperature(model, X_cal, y_cal):
    model.eval()
    logits = []
    with torch.no_grad():
        for i in range(0, len(y_cal), 1024):
            xb = torch.from_numpy(X_cal[i:i + 1024]).permute(0, 3, 1, 2).to(device)
            logits.append(model(xb).detach().cpu().numpy())
    logits = np.concatenate(logits).astype(np.float64)
    yc = np.asarray(y_cal, dtype=np.int64)

    def nll(t):
        t = max(float(t), 1e-3)
        z = logits / t
        zm = z - z.max(1, keepdims=True)
        logsum = np.log(np.exp(zm).sum(1)) + z.max(1)
        return float(-np.mean(z[np.arange(len(yc)), yc] - logsum))

    best_t, best_v = 1.0, nll(1.0)
    for t in np.linspace(0.1, 5.0, 50):
        v = nll(t)
        if v < best_v:
            best_t, best_v = t, v
    for t in np.linspace(max(0.1, best_t - 0.5), best_t + 0.5, 200):
        v = nll(t)
        if v < best_v:
            best_t, best_v = t, v
    return best_t

cal_X, cal_y = X_va, y_va
phm = src_va == "photo"
if phm.any():
    cal_X, cal_y = X_va[phm], y_va[phm]
    print(f"fitting temperature on {len(cal_y)} PHOTO val cells")
else:
    print(f"fitting temperature on {len(cal_y)} val cells")
T = fit_temperature(model, cal_X, cal_y)
print(f"temperature scaling: T = {T:.3f}")
with open(os.path.join(DRIVE_DIR, "digit_cnn.pth.temperature.json"), "w",
          encoding="utf-8") as f:
    json.dump({"temperature": T}, f)
print("saved calibration to", os.path.join(DRIVE_DIR,
                                           "digit_cnn.pth.temperature.json"))


In [ ]:
# --- 10c. OPTIONAL: resume from the latest checkpoint ---------------------------
# If the runtime died mid-training, run this cell, then re-run the training cell
# above: it loads the newest Drive checkpoint (model, optimizer, history) and
# continues. Only runs when YOU run this cell - training never auto-resumes.
def latest_checkpoint(ckpt_dir):
    ckpts = [f for f in os.listdir(ckpt_dir) if re.match(r"ckpt_epoch\d+\.pt$", f)]
    return os.path.join(ckpt_dir, sorted(ckpts)[-1]) if ckpts else None


if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')
ckpt_dir = os.path.join(CONFIG["drive_dir"], "checkpoints")
CONFIG["ckpt_dir"] = ckpt_dir
RESUME_PATH = None
latest = latest_checkpoint(ckpt_dir)
if latest:
    RESUME_PATH = latest
    print("Set RESUME_PATH =", latest, "- now re-run the training cell above.")
else:
    print("No checkpoints found yet in", ckpt_dir)


## 11. Final evaluation on the held-out test split

The test slice has never been seen during training. We report:

* overall test accuracy,
* per-digit accuracy,
* accuracy by **source** (photo / synthetic / mnist - each domain separately),
* a confusion matrix,
* the training curves.


In [ ]:
# --- 11. Evaluation on the held-out test split ---------------------------------
def evaluate(model, X, y, device, batch_size=256):
    """Batched inference on the full array (avoid OOM on large test sets)."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i + batch_size]).permute(0, 3, 1, 2).to(device)
            yb = torch.from_numpy(y[i:i + batch_size]).to(device)
            out = model(xb)
            preds.append(out.argmax(1).cpu().numpy())
    pred = np.concatenate(preds) if preds else np.array([], dtype=np.int64)
    return float((pred == y).mean()), pred


# Official test = MNIST/EMNIST TEST splits only (never in the training pool;
# the random split above is for train/val and a dev-test slice). If no
# handwriting source is enabled, fall back to the dev-test slice.
if len(y_mnist_te) or len(y_emnist_te):
    X_te = np.concatenate([X_mnist_te, X_emnist_te]).astype(np.float32)
    y_te = np.concatenate([y_mnist_te, y_emnist_te]).astype(np.int64)
    src_te = np.array(["mnist"] * len(y_mnist_te) + ["emnist"] * len(y_emnist_te))
    print("official test set: mnist", len(y_mnist_te), "| emnist", len(y_emnist_te))
else:
    print("official test set: none (no handwriting sources) - using dev-test slice")

test_acc, test_pred = evaluate(model, X_te, y_te, device)
print(f"test accuracy: {test_acc:.4f}")

print("\nper-class accuracy:")
per_cls = {}
for k in np.unique(y_te):
    m = y_te == k
    a = float((test_pred[m] == k).mean())
    per_cls[int(k)] = a
    print(f"  digit {int(k)}: {a:.4f}  ({m.sum()} samples)")
print(f"  macro avg: {np.mean(list(per_cls.values())):.4f}")

print("\naccuracy by source:")
per_src = {}
for s in np.unique(src_te):
    m = src_te == s
    a = float((test_pred[m] == y_te[m]).mean())
    per_src[s] = a
    print(f"  {s:9s}: {a:.4f}  ({m.sum()} samples)")
print(f"  macro avg (sources): {np.mean(list(per_src.values())):.4f}")

# --- visual verification: easy to eyeball model performance ------------------
lab = np.array([f"{t}->{p}" for t, p in zip(y_te, test_pred)])
show_montage(X_te, lab, n=24, seed=SEED, classes=y_te,
             title="held-out test: each label is TRUE->PREDICTED")
mis = np.where(test_pred != y_te)[0]
if len(mis):
    show_montage(X_te[mis], lab[mis], n=min(24, len(mis)), seed=SEED,
                 classes=y_te[mis],
                 title=f"misclassified samples ({len(mis)} total; TRUE->PREDICTED)")

cm = np.zeros((10, 10), dtype=int)
for p, t in zip(test_pred, y_te):
    cm[t, p] += 1
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xlabel("predicted")
plt.ylabel("true")
plt.xticks(range(10), range(10))
plt.yticks(range(10), range(10))
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.title("confusion matrix (test split)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(hist["train_loss"], label="train")
plt.plot(hist["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("loss")
plt.subplot(1, 2, 2)
plt.plot(hist["val_acc"], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.title("validation accuracy")
plt.tight_layout()
plt.show()


## 12. Export the trained model

`digit_cnn.pth` is a plain `state_dict` - exactly what the repo's
`load_digit_model()` expects. Download it and place it next to `digit_cnn.py`:

```python
from digit_cnn import load_digit_model
model = load_digit_model("digit_cnn.pth")
```


In [ ]:
# --- 12. Export digit_cnn.pth ---------------------------------------------------
# The exported file MUST be the best checkpoint, never the last epoch's weights.
bp = os.path.join(CONFIG["ckpt_dir"], "best_model.pt")
if os.path.exists(bp):
    model.load_state_dict(torch.load(bp, map_location=device)["model_state_dict"])
    print("loaded best_model.pt (authoritative weights for export)")
torch.save(model.state_dict(), "digit_cnn.pth")
print("saved digit_cnn.pth")

from google.colab import files
files.download("digit_cnn.pth")

# keep a copy on Google Drive (mounted by the training cell) so the weights
# survive runtime resets and can be pulled from Drive without a re-download
if os.path.exists("/content/drive"):
    import shutil
    drive_dst = os.path.join(CONFIG["drive_dir"], "digit_cnn.pth")
    shutil.copy("digit_cnn.pth", drive_dst)
    print("copied digit_cnn.pth to", drive_dst)

# The temperature sidecar (written by cell 10b) must sit NEXT TO digit_cnn.pth
# for the repo's load_temperature() to find it - download both files.
if os.path.exists(os.path.join(CONFIG["drive_dir"], "digit_cnn.pth.temperature.json")):
    files.download(os.path.join(CONFIG["drive_dir"], "digit_cnn.pth.temperature.json"))
print("download digit_cnn.pth AND digit_cnn.pth.temperature.json - "
      "the repo reads both (load_digit_model + load_temperature)")


## 13. Benchmark: end-to-end performance on labeled sudoku puzzles

**This section is the official benchmark** of the whole pipeline (grid detection ->
preprocessing -> CNN recognition -> solve). Photo data may be used for TRAINING
(section 7) under the sealed split, so the authoritative numbers are measured on
the **sealed TEST partition** (`CONFIG["hf_full_test"] = True`) - photos the model
has never seen. The default lightweight 24-image sample is a quick smoke check.

Every puzzle set is evaluated **twice, in two explicitly-labeled modes**:

1. **RECOGNITION/SOLVER** - the dataset's ground-truth grid corners are warped
   directly. This isolates CNN recognition + solving from grid detection: the
   headline **solved-cell DIGIT accuracy** (plus empty accuracy, definite-cell
   accuracy, exact grids and solve rate) is measured with detection out of the
   picture.
2. **END-TO-END** - the corners are ignored and the grid is found automatically
   (contour -> lines). Detection misses are counted honestly in a
   **grid-detection rate** (there is no blind center-crop fallback that would
   inflate the number) and failed puzzles are excluded from the recognition
   metrics. This is the real `photo -> detect -> solve` number.

**Primary: Lexski/sudoku-image-recognition** (`CONFIG["hf_benchmark"]`, default on)
- 1400 images with explicit train/val/test splits. By default the benchmark
  **downloads only a seeded sample of `CONFIG["hf_sample_size"]` images (default 24)**
  straight from the HF repo with plain urllib - fast, a few MB, no `datasets`
  library. With `CONFIG["hf_full_test"] = True` it instead benchmarks the FULL
  **sealed test partition** (210 puzzles, the same names as the repo's
  `photo_splits.json` - requires the metadata downloaded, and the partition is
  rebuilt with the same seed 42). Ground truth is per-cell flags: a cell is
  *definite* when solved with exactly one digit, or unsolved with no candidates
  (empty); candidate-only cells are skipped. Keypoints reordered
  TL,BL,BR,TR -> TL,TR,BR,BL.

**Fallback: wichtounet folder** (the provisioned clone, `labeled_eval_frac` share)
when the HF dataset cannot be loaded. Its version (V2 givens vs mixed complete
grids) is detected automatically so the two are never mixed.


In [ ]:
# --- 13. OPTIONAL: point at your sudoku photos ----------------------------------
# Either set CONFIG["cpecog_dir"] above to an existing folder, or upload cpecog.zip
# right here (uncomment, run, and it will extract into /content/cpecog):
# from google.colab import files
# for name, _ in files.upload().items():
#     os.system(f"unzip -q -o {name} -d /content/cpecog")
# CONFIG["cpecog_dir"] = "/content/cpecog"
print("cpecog_dir:", CONFIG["cpecog_dir"])


In [ ]:
# --- 13b. Verification run -------------------------------------------------------
# Port of sudoku_core.py's grid detection + solver-guided error correction.

def four_point_transform(img, pts, size=600):
    rect = order_points(pts)
    dst = np.array([[0, 0], [size - 1, 0], [size - 1, size - 1], [0, size - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(img, M, (size, size))


def detect_grid_contour(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 15, 5)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    biggest = max(cnts, key=cv2.contourArea)
    peri = cv2.arcLength(biggest, True)
    approx = cv2.approxPolyDP(biggest, 0.02 * peri, True)
    return approx if len(approx) == 4 else None


def detect_grid_lines(gray, line_gap=12, min_cluster=3):
    h, w = gray.shape
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    bg = float(np.percentile(blur, 90))
    line_mask = ((blur < max(90, bg - 40)) & (blur < 235)).astype(np.uint8) * 255
    klen = max(w // 12, 1)
    hlines = cv2.morphologyEx(line_mask, cv2.MORPH_OPEN,
                              cv2.getStructuringElement(cv2.MORPH_RECT, (klen, 1)))
    vlines = cv2.morphologyEx(line_mask, cv2.MORPH_OPEN,
                              cv2.getStructuringElement(cv2.MORPH_RECT, (1, klen)))

    def locate(binmap, axis):
        dens = binmap.mean(axis=axis)
        idxs = np.where(dens > 0.05)[0]
        clusters = []
        for i in idxs:
            if clusters and i - clusters[-1][-1] <= line_gap:
                clusters[-1].append(i)
            else:
                clusters.append([i])
        return [int(np.mean(c)) for c in clusters if len(c) >= min_cluster]

    return locate(hlines, axis=1), locate(vlines, axis=0)


def line_grid_quad(gray, size=600):
    h, v = detect_grid_lines(gray)
    if h is None or len(h) < 9 or len(v) < 9:
        return None
    h, v = sorted(h), sorted(v)
    dh, dv = np.median(np.diff(h)), np.median(np.diff(v))
    # Line spacing is measured in SOURCE pixels (the grid spans roughly the
    # whole source image), NOT against the requested output `size` - the old
    # size-based bounds rejected valid 300 px and 1200 px images.
    ref = max(min(gray.shape), 1)
    if not (ref * 0.06 < dh < ref * 0.14 and ref * 0.06 < dv < ref * 0.14):
        return None
    y0 = max(0, int(h[0] - dh / 2)); y1 = min(gray.shape[0], int(h[-1] + dh / 2))
    x0 = max(0, int(v[0] - dv / 2)); x1 = min(gray.shape[1], int(v[-1] + dv / 2))
    src = np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], dtype="float32")
    dst = np.array([[0, 0], [size - 1, 0], [size - 1, size - 1], [0, size - 1]], dtype="float32")
    return cv2.warpPerspective(gray, cv2.getPerspectiveTransform(src, dst), (size, size))


def classify_cells(model, cells, device="cpu"):
    """81 row-major cells -> (81,10) softmax; class 0 = empty cell."""
    batch = np.stack([preprocess_cell(c) for c in cells])
    return classify_inputs(batch, model, device)


def classify_inputs(inputs, model, device="cpu"):
    """(N,48,48,1) already-preprocessed inputs -> (N,10) softmax."""
    model.eval()
    with torch.no_grad():
        xb = torch.from_numpy(np.asarray(inputs, dtype=np.float32)).permute(0, 3, 1, 2).to(device)
        return torch.softmax(model(xb), dim=1).cpu().numpy()


def _finish_cell(th, target=48):
    """Phase 1 component cleanup + letterboxed resize -> (48,48,1) float32.

    Same algorithm as the repo's digit_cnn._finish_preprocess (v5): 10%
    margin strip, tiny-component drop (< 0.5% of the FULL cell),
    shape-based line-fragment removal (edge-to-edge lines up to 6 px thin +
    elongated; one-border lines < 2.5 px thin spanning >= 70%; L-shaped
    corner remnants), split strokes re-joined with a THIN BRIDGE, ALL
    surviving components kept (no largest-component-only), conservative
    empty (largest surviving < 1% -> all-black), letterboxed
    aspect-preserving resize.
    """
    h, w = th.shape
    m = max(1, int(round(min(h, w) * 0.10)))
    th[:m, :] = 0
    th[-m:, :] = 0
    th[:, :m] = 0
    th[:, -m:] = 0
    inner = th[m:h - m, m:w - m]
    hh, ww = inner.shape
    min_area = h * w * 0.005
    if (inner > 0).any():
        n, labels, cc, _ = cv2.connectedComponentsWithStats(inner, 8)
        for i in range(1, n):
            x, y, cw, chh, area = cc[i, 0], cc[i, 1], cc[i, 2], cc[i, 3], cc[i, 4]
            if area < min_area:
                inner[labels == i] = 0
                continue
            length = max(cw, chh)
            thin = area / max(length, 1) < 6.0
            elongated = length / max(min(cw, chh), 1) > 4.0
            if ((x == 0 and x + cw == ww) or (y == 0 and y + chh == hh)) and thin and elongated:
                inner[labels == i] = 0
                continue
            touch = (x == 0 or x + cw == ww or y == 0 or y + chh == hh)
            thin2 = area / max(length, 1) < 2.5
            if touch and thin2 and elongated and length >= 0.7 * max(hh, ww):
                inner[labels == i] = 0
                continue
            if (((x == 0 or x + cw == ww) and (y == 0 or y + chh == hh)) and
                    cw >= 0.4 * ww and chh >= 0.4 * hh and area < 0.06 * h * w):
                inner[labels == i] = 0
    for _ in range(4):
        n, labels, cc, _ = cv2.connectedComponentsWithStats(inner, 8)
        if n <= 2:
            break
        comps = [(i, cc[i, 0], cc[i, 1], cc[i, 2], cc[i, 3], cc[i, 4]) for i in range(1, n)]
        best = None
        for ai in range(len(comps)):
            for bi in range(ai + 1, len(comps)):
                ia, xa, ya, wa, ha, aa = comps[ai]
                ib, xb, yb, wb, hb, ab = comps[bi]
                if aa < min_area or ab < min_area:
                    continue
                if ya + ha <= yb:
                    gap, ov, ov_ref = yb - (ya + ha), min(xa + wa, xb + wb) - max(xa, xb), min(wa, wb)
                    bx = (max(xa, xb) + min(xa + wa, xb + wb)) // 2
                    (x1, y1), (x2, y2) = (bx, ya + ha), (bx, yb)
                elif yb + hb <= ya:
                    gap, ov, ov_ref = ya - (yb + hb), min(xa + wa, xb + wb) - max(xa, xb), min(wa, wb)
                    bx = (max(xa, xb) + min(xa + wa, xb + wb)) // 2
                    (x1, y1), (x2, y2) = (bx, yb + hb), (bx, ya)
                elif xa + wa <= xb:
                    gap, ov, ov_ref = xb - (xa + wa), min(ya + ha, yb + hb) - max(ya, yb), min(ha, hb)
                    by = (max(ya, yb) + min(ya + ha, yb + hb)) // 2
                    (x1, y1), (x2, y2) = (xa + wa, by), (xb, by)
                elif xb + wb <= xa:
                    gap, ov, ov_ref = xa - (xb + wb), min(ya + ha, yb + hb) - max(ya, yb), min(ha, hb)
                    by = (max(ya, yb) + min(ya + ha, yb + hb)) // 2
                    (x1, y1), (x2, y2) = (xb + wb, by), (xa, by)
                else:
                    continue
                if gap < 0 or gap > 5 or ov < 0.5 * ov_ref:
                    continue
                bx0, bx1 = min(xa, xb), max(xa + wa, xb + wb)
                by0, by1 = min(ya, yb), max(ya + ha, yb + hb)
                aspect = max(bx1 - bx0, by1 - by0) / max(min(bx1 - bx0, by1 - by0), 1)
                if aspect <= 3.5 and aa + ab >= min_area:
                    if best is None or aa + ab > best[0]:
                        best = (aa + ab, ia, ib, x1, y1, x2, y2)
        if best is None:
            break
        _, _, _, x1, y1, x2, y2 = best
        cv2.line(inner, (x1, y1), (x2, y2), 255, 2)
    n, labels, cc, _ = cv2.connectedComponentsWithStats(inner, 8)
    if n > 1:
        areas = cc[1:, 4]
        if areas.max() < h * w * 0.01:       # conservative empty: nothing
            inner[:] = 0                     # credible survived (< 1% full cell)
    ys, xs = np.nonzero(inner)
    if len(xs) == 0:
        return np.zeros((target, target, 1), dtype=np.float32)
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    pad = int(max(x1 - x0, y1 - y0) * 0.12) + 2
    crop = inner[max(0, y0 - pad):y1 + pad + 1, max(0, x0 - pad):x1 + pad + 1]
    ch, cw = crop.shape
    scale = target / max(ch, cw)
    nh, nw = max(1, int(round(ch * scale))), max(1, int(round(cw * scale)))
    resized = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target, target), dtype=np.float32)
    y0c, x0c = (target - nh) // 2, (target - nw) // 2
    canvas[y0c:y0c + nh, x0c:x0c + nw] = resized
    return (canvas / 255.0)[..., None]


def preprocess_variants(cell, target=48):
    """Re-sensing views of a raw cell: adaptive C=5/7/9 + Otsu (polarity
    auto-corrected), each finished by the Phase 1 `_finish_cell` cleanup
    (margin strip, fragment removal, conditional merge, letterbox)."""
    if cell.ndim == 3:
        cell = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    if cell.dtype != np.uint8:
        cell = np.clip(cell, 0, 255).astype(np.uint8)
    blur = cv2.GaussianBlur(cell, (3, 3), 0)
    views = []
    for c_val in (5, 7, 9):
        th = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 15, c_val)
        views.append(_finish_cell(th, target))
    otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    if (otsu > 0).mean() > 0.5:
        otsu = 255 - otsu
    views.append(_finish_cell(otsu, target))
    return views


def find_violated_cells(grid):
    """-> (suspects, all_violated). suspects = cells holding a digit that
    appears more than once in a row/column/box; all_violated = every cell of
    any violated unit."""
    suspects = set()
    bad = set()

    def check_unit(idx_list):
        vals = {}
        for idx in idx_list:
            v = grid[idx // 9, idx % 9]
            if v:
                vals.setdefault(v, []).append(idx)
        if any(len(l) > 1 for l in vals.values()):
            bad.update(idx_list)
            for l in vals.values():
                if len(l) > 1:
                    suspects.update(l)

    for r in range(9):
        check_unit([r * 9 + c for c in range(9)])
    for c in range(9):
        check_unit([r * 9 + c for r in range(9)])
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            check_unit([(br + i) * 9 + (bc + j) for i in range(3) for j in range(3)])
    return suspects, bad


def solve_with_resensing(grid, probs, cells, classify_fn, max_correction_cells=12,
                         max_rounds=2, conf_thresh=0.9, max_nodes=200_000):
    """Constraint-guided re-sensing solve (same algorithm as sudoku_core.py).

    Tries the plain bounded correction first; if the puzzle does not solve,
    re-senses the SUSPECTS (cells holding duplicated digits) plus the
    least-confident cells through preprocessing variants (adaptive C=5/7/9 +
    Otsu), averages the new opinions with the first pass, and retries.
    Returns (solution_grid, ok, n_resensed).
    """
    base = grid.copy()
    first_pass = grid.copy()
    cur_probs = probs.copy()
    resensed = set()
    total_resensed = 0
    for round_no in range(max_rounds + 1):
        cap = max_correction_cells * (round_no + 1)
        suspects, violated = find_violated_cells(base)
        # re-sense only when there is something to disambiguate: violations
        # exist (round 0) or earlier rounds failed (round_no > 0)
        if round_no > 0 or suspects:
            focus = list(suspects)
            for i in range(81):
                if len(focus) >= cap:
                    break
                if i not in focus and cur_probs[i].max() < conf_thresh:
                    focus.append(i)
            focus.sort(key=lambda i: (0 if i in suspects else 1, cur_probs[i].max()))
            focus = focus[:cap]
            for idx in focus:
                if idx in resensed:
                    continue
                views = np.stack(preprocess_variants(cells[idx])).astype(np.float32)
                vp = np.asarray(classify_fn(views), dtype=np.float64).reshape(-1, 10)
                # conservative blend: first-pass opinion stays dominant
                cur_probs[idx] = 0.7 * cur_probs[idx] + 0.3 * vp.mean(axis=0)
                # the working puzzle follows the updated opinion - a changed
                # argmax must actually reach the next correction round
                base.flat[idx] = int(cur_probs[idx].argmax())
                resensed.add(idx)
                total_resensed += 1

        rank = {i: 0 for i in suspects}
        rank.update({i: 1 for i in violated if i not in rank})
        candidates, order = candidates_from_probs(cur_probs, max_alts=3,
                                                  conf_thresh=conf_thresh,
                                                  order_cap=cap,
                                                  priority=rank)
        solved, ok = solve_with_correction(base, candidates, order,
                                           max_nodes=max_nodes,
                                           budget_ref=first_pass)
        if ok:
            return solved, True, total_resensed
        if round_no == max_rounds:
            break
    return base, False, total_resensed


def sudoku_from_cells(model, cells, device="cpu"):
    """81 row-major cells -> 9x9 grid (class 0 = empty cell, learned from
    real photo empties during training - no hand-tuned ink gate)."""
    return classify_cells(model, cells, device).argmax(1).reshape(9, 9)


def hf_cells_to_gt(cells):
    """HF (9,9,10) flags -> (gt_grid, definite_mask).

    flag 0 = solved; flags 1-9 = digit presence. Definite cells are solved
    cells with exactly one digit, and unsolved cells with no candidates
    (empty). Cells with candidates are NOT definite (skipped in accuracy).
    """
    cells = np.asarray(cells).reshape(9, 9, 10)
    gt = np.zeros((9, 9), dtype=int)
    definite = np.zeros((9, 9), dtype=bool)
    for r in range(9):
        for c in range(9):
            flags = cells[r, c]
            solved = int(flags[0]) == 1
            digits = [d for d in range(1, 10) if int(flags[d]) == 1]
            if solved and len(digits) == 1:
                gt[r, c] = digits[0]
                definite[r, c] = True
            elif not solved and len(digits) == 0:
                gt[r, c] = 0
                definite[r, c] = True
    return gt, definite


def candidates_from_probs(probs, max_alts=3, conf_thresh=0.9, order_cap=12,
                          priority=None):
    """Build (candidates, order) for solve_with_correction from CNN softmax.

    probs: (81, 10) softmax rows; class 0 = empty cell.
    candidates: {cell_index: [alternatives, best first]}; the current argmax
        guess is always first, the rest are the next-best classes (including
        class 0, so correction may also *remove* a wrongly recognized digit).
    order: the order_cap cell indices to reconsider, least confident first.
        `priority` is a {cell_index: rank} map (lower = earlier) - e.g. the
        suspects/violated cells of re-sensing.
    """
    candidates = {}
    order = []
    for idx, p in enumerate(probs):
        guess = int(p.argmax())
        ranked = [int(k) for k in p.argsort()[::-1]]
        if priority and priority.get(idx, 2) == 0:
            # suspect (rank 0): digit alternatives first, then removal (empty),
            # current value LAST - prefer a real correction over deleting givens
            others = [k for k in ranked if k != guess and k != 0][:2]
            alts = (others + [0, guess]) if guess != 0 else ([0] + others)
        else:
            alts = [guess] + [k for k in ranked if k != guess][:max_alts - 1]
        candidates[idx] = alts
        if p.max() < conf_thresh or (priority and idx in priority):
            order.append(idx)
    order.sort(key=lambda idx: (priority.get(idx, 2) if priority else 1,
                                probs[idx].max()))
    if order_cap:
        order = order[:order_cap]
    return candidates, order


_DIGIT_BITS = 0x3FE        # candidate bits 1..9; bit 0 = "release" marker


def _search(grid, domains, branch_order, max_nodes, accept_fn=None):
    """Shared backtracking search engine with constraint propagation (the
    same engine as sudoku_core._search).

    grid: 9x9 int working copy (0 = empty); filled with the solution on
        success. domains: 9x9 bitmasks - bit d means digit d is still allowed;
        bit 0 marks a cell the correction path may 'release' (treat as a
        plain empty and fill it freely). branch_order: {cell_index: [digit,
        ...]} tried best-first. max_nodes: total branching-node budget.
        accept_fn: optional validator on every complete grid.

    Assignment legality is checked against 9-bit row/col/box used-masks;
    propagation (naked + hidden singles) runs at every node; MRV branching
    only when no cell is forced. When propagation dead-ends the search UNDOES
    the partial fills and branches on a releasable order cell first (release
    = its first action when the CNN guessed empty) - propagation used to
    force a restricted correction domain's digit, stranding the true digit
    and failing exact-recognition puzzles whose candidate list missed it.
    """
    DIGITS = 0x3FE
    grid = grid.copy()
    domains = [[int(domains[r, c]) for c in range(9)] for r in range(9)]
    row = [0] * 9
    col = [0] * 9
    box = [0] * 9
    for r in range(9):
        for c in range(9):
            v = int(grid[r, c])
            if v:
                bit = 1 << v
                row[r] |= bit
                col[c] |= bit
                box[3 * (r // 3) + c // 3] |= bit
    units = []
    for r in range(9):
        units.append(([(r, c) for c in range(9)], row, r))
    for c in range(9):
        units.append(([(r, c) for r in range(9)], col, c))
    for br in range(3):
        for bc in range(3):
            units.append(([(br * 3 + i, bc * 3 + j) for i in range(3) for j in range(3)],
                          box, 3 * br + bc))
    trail = []
    nodes = 0
    aborted = False

    def cand(r, c):
        return domains[r][c] & ~(row[r] | col[c] | box[3 * (r // 3) + c // 3])

    def fill(r, c, v):
        grid[r, c] = v
        bit = 1 << v
        row[r] |= bit
        col[c] |= bit
        box[3 * (r // 3) + c // 3] |= bit
        trail.append(("f", r, c))

    def undo(mark):
        while len(trail) > mark:
            t = trail.pop()
            if t[0] == "f":
                _, r, c = t
                v = int(grid[r, c])
                grid[r, c] = 0
                bit = 1 << v
                row[r] &= ~bit
                col[c] &= ~bit
                box[3 * (r // 3) + c // 3] &= ~bit
            else:
                _, r, c, old = t
                domains[r][c] = old

    def propagate():
        while True:
            filled_any = False
            for r in range(9):
                for c in range(9):
                    if grid[r, c]:
                        continue
                    dbits = cand(r, c) & DIGITS
                    if not dbits:
                        return False
                    if dbits & (dbits - 1) == 0:
                        fill(r, c, dbits.bit_length() - 1)
                        filled_any = True
            if filled_any:
                continue
            for cells, mask_list, mi in units:
                umask = mask_list[mi]
                for d in range(1, 10):
                    bit = 1 << d
                    if umask & bit:
                        continue
                    where = None
                    cnt = 0
                    for r, c in cells:
                        if not grid[r, c] and cand(r, c) & bit:
                            cnt += 1
                            where = (r, c)
                            if cnt > 1:
                                break
                    if cnt == 0:
                        return False
                    if cnt == 1:
                        fill(where[0], where[1], d)
                        filled_any = True
            if not filled_any:
                return True

    def dfs():
        nonlocal nodes, aborted
        if aborted:
            return False
        nodes += 1
        if nodes > max_nodes:
            aborted = True
            return False
        mark0 = len(trail)
        prop_ok = propagate()
        if not prop_ok:
            undo(mark0)
        best = None
        best_cnt = 10
        if not prop_ok:
            # Branch on a releasable order cell first (release = its first
            # action when the CNN guessed empty): expanding its domain to the
            # full DIGITS set is what repairs the contradiction. A cell whose
            # candidate list merely CONTAINS 0 (suspects keep 0 last so the
            # search prefers a real correction) can also repair it - fall
            # back to those when no release-first cell is free.
            for idx, alts in branch_order.items():
                if alts and alts[0] == 0:
                    r, c = divmod(idx, 9)
                    if grid[r, c] == 0:
                        best, best_cnt = (r, c), -1
                        break
            if best is None:
                for idx, alts in branch_order.items():
                    if 0 in alts:
                        r, c = divmod(idx, 9)
                        if grid[r, c] == 0:
                            best, best_cnt = (r, c), -1
                            break
        if best is None:
            for r in range(9):
                for c in range(9):
                    if grid[r, c]:
                        continue
                    cnt = (cand(r, c) & DIGITS).bit_count()
                    if cnt < best_cnt:
                        best_cnt = cnt
                        best = (r, c)
                        if cnt == 1:
                            break
                if best_cnt == 1:
                    break
        if best is None:
            return accept_fn is None or accept_fn(grid)
        r, c = best
        cnd = cand(r, c)
        ordered = branch_order.get(r * 9 + c)
        if ordered is not None:
            for v in ordered:
                mark = len(trail)
                if v == 0:
                    if not (cnd & 1):
                        continue
                    old = domains[r][c]
                    domains[r][c] = DIGITS
                    trail.append(("d", r, c, old))
                else:
                    if not (cnd & (1 << v)):
                        continue
                    fill(r, c, v)
                if dfs():
                    return True
                undo(mark)
        else:
            for d in range(1, 10):
                if cnd & (1 << d):
                    mark = len(trail)
                    fill(r, c, d)
                    if dfs():
                        return True
                    undo(mark)
        return False

    ok = dfs()
    if not ok:
        undo(0)
    return grid, ok


def solve_sudoku(grid, max_nodes=100_000):
    """Constraint-propagation MRV solver -> (solved_grid, ok).

    Bitmask candidate domains, naked/hidden-single propagation at every node,
    most-constrained-cell branching. Duplicate givens are rejected upfront.
    """
    grid = grid.copy().astype(int)

    def valid_givens(g):
        for r in range(9):
            seen = [v for v in g[r, :] if v]
            if len(set(seen)) != len(seen):
                return False
        for c in range(9):
            seen = [v for v in g[:, c] if v]
            if len(set(seen)) != len(seen):
                return False
        for br in range(0, 9, 3):
            for bc in range(0, 9, 3):
                seen = [v for v in g[br:br + 3, bc:bc + 3].flat if v]
                if len(set(seen)) != len(seen):
                    return False
        return True

    if not valid_givens(grid):
        return grid, False
    domains = np.where(grid != 0, 1 << grid, _DIGIT_BITS).astype(int)
    return _search(grid, domains, {}, max_nodes)


def solve_with_correction(grid, candidates, order, max_nodes=200_000,
                          change_budget=4, budget_ref=None):
    """Confidence-aware error correction as ONE integrated search (the same
    algorithm as sudoku_core.solve_with_correction).

    grid: best-guess puzzle (0 = empty cell).
    candidates: dict cell_index -> list of digit alternatives, best first
        (class 0 = release the cell, letting the solver fill it freely - a
        wrongly recognized digit gets replaced).
    order: cell indices to reconsider, least confident first.
    change_budget: accept a solution only if at most this many RECOGNIZED
        DIGITS differ (empty cells filled by the solver do not count) -
        anchors the solution to the photo.
    budget_ref: the FIRST-PASS recognized grid used for the budget; defaults
        to `grid`. Re-sensing updates `grid` between rounds while the budget
        keeps counting edits against what the CNN first saw.
    Returns (solution_grid, ok).
    """
    base = grid.copy().astype(int)
    ref = (budget_ref if budget_ref is not None else base).copy()
    order_set = set(order)
    for idx in order:
        r, c = divmod(idx, 9)
        base[r, c] = 0                    # variable cells start empty

    def fixed_units_ok():
        def unit_ok(cells):
            seen = set()
            for idx in cells:
                if idx in order_set:
                    continue
                v = base.flat[idx]
                if v:
                    if v in seen:
                        return False
                    seen.add(v)
            return True

        for r in range(9):
            if not unit_ok([r * 9 + c for c in range(9)]):
                return False
        for c in range(9):
            if not unit_ok([r * 9 + c for r in range(9)]):
                return False
        for br in range(0, 9, 3):
            for bc in range(0, 9, 3):
                if not unit_ok([(br + i) * 9 + (bc + j) for i in range(3) for j in range(3)]):
                    return False
        return True

    if not fixed_units_ok():
        return base, False

    domains = np.full((9, 9), _DIGIT_BITS, dtype=int)
    branch_order = {}
    for idx in order:
        r, c = divmod(idx, 9)
        alts = candidates[idx]
        mask = 0
        for v in alts:
            mask |= 1 << v
        if not mask & _DIGIT_BITS:
            mask = _DIGIT_BITS          # release-only cell: fill freely
        domains[r, c] = mask
        branch_order[idx] = alts
    for idx in range(81):
        if idx not in order_set:
            v = base.flat[idx]
            if v:
                r, c = divmod(idx, 9)
                domains[r, c] = 1 << v

    def accept(g):
        return sum(1 for idx in order
                   if ref.flat[idx] != 0 and g.flat[idx] != ref.flat[idx]) <= change_budget

    return _search(base, domains, branch_order, max_nodes, accept)


def valid_solution(g):
    for r in range(9):
        if sorted(g[r, :]) != list(range(1, 10)):
            return False
    for c in range(9):
        if sorted(g[:, c]) != list(range(1, 10)):
            return False
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            if sorted(g[br:br + 3, bc:bc + 3].flat) != list(range(1, 10)):
                return False
    return True


In [ ]:
# --- 13c. Run the benchmark -----------------------------------------------------
# Evaluates a list of puzzle dicts {name, img(BGR), kp(TL,TR,BR,BL) or None,
# gt, definite}: warp -> extract -> classify -> compare -> bounded-correction
# solve. Wrong cells are marked with *; candidate-only cells show as '.'.
#
# Two benchmark modes are reported separately:
#   use_detection=False -> RECOGNITION/SOLVER: the ground-truth grid corners
#                          are warped directly (isolates CNN + solver from
#                          grid detection);
#   use_detection=True  -> END-TO-END: corners are ignored and the grid is
#                          found automatically (contour -> lines). Detection
#                          misses stay in the detection-rate denominator but
#                          are excluded from the recognition metrics.
def detect_and_warp(gray, size=600):
    # no blind center-crop fallback: a failed detection is a real miss
    quad = detect_grid_contour(gray)
    if quad is not None:
        return four_point_transform(gray, quad, size), "contour"
    warped = line_grid_quad(gray, size)
    if warped is not None:
        return warped, "lines"
    return None, None


def eval_puzzle_set(puzzles, title, use_detection=False):
    n_total = len(puzzles)
    n_detected = n_exact = n_solved = 0
    total_resensed = 0
    cor_def = tot_def = 0
    cor_digit = tot_digit = 0
    cor_empty = tot_empty = 0
    per_digit_cor = {k: [0, 0] for k in range(1, 10)}
    examples = []
    for pz in puzzles:
        gray = cv2.cvtColor(pz["img"], cv2.COLOR_BGR2GRAY)
        if use_detection:
            warped, method = detect_and_warp(gray)
            if warped is None:
                continue                     # detection miss (counted below)
        else:
            warped = four_point_transform(gray, pz["kp"])
        n_detected += 1
        cells = extract_cells(warped)
        probs = classify_cells(model, cells, device)
        grid = probs.argmax(1).reshape(9, 9)
        gt = pz["gt"]
        definite = pz["definite"]
        if definite is None:
            definite = np.ones((9, 9), dtype=bool)
        tot_def += int(definite.sum())
        cor_def += int(((grid == gt) & definite).sum())
        dig = definite & (gt != 0)
        emp = definite & (gt == 0)
        tot_digit += int(dig.sum())
        cor_digit += int((grid[dig] == gt[dig]).sum())
        tot_empty += int(emp.sum())
        cor_empty += int((grid[emp] == 0).sum())
        for k in range(1, 10):
            m = dig & (gt == k)
            per_digit_cor[k][1] += int(m.sum())
            per_digit_cor[k][0] += int((grid[m] == k).sum())
        if ((grid == gt) | ~definite).all():
            n_exact += 1
        solved, ok, n_resensed = solve_with_resensing(
            grid, probs, cells,
            lambda views: classify_inputs(views, model, device),
            max_correction_cells=CONFIG["max_correction_cells"])
        total_resensed += n_resensed
        if ok and valid_solution(solved):
            n_solved += 1
        if len(examples) < 2:
            examples.append((pz["name"], grid, gt, definite, solved, ok))
    mode_label = "END-TO-END (auto grid detection)" if use_detection else \
        "RECOGNITION/SOLVER (ground-truth corners)"
    nd = max(n_detected, 1)
    print(f"[{title}] mode: {mode_label}")
    print(f"  evaluated:        {n_detected}/{n_total} puzzles"
          f" | cells re-sensed: {total_resensed}")
    if use_detection and n_total:
        print(f"  grid detection:   {n_detected}/{n_total} ({n_detected / n_total:.1%})")
    print(f"  definite-cell acc: {cor_def / max(tot_def, 1):.4f}")
    print(f"  DIGIT acc (solved): {cor_digit / max(tot_digit, 1):.4f}  "
          f"<-- number-recognition headline")
    print(f"  empty acc:          {cor_empty / max(tot_empty, 1):.4f}")
    print(f"  exact grids:        {n_exact}/{n_detected} ({n_exact / nd:.1%})")
    print(f"  solve rate:         {n_solved}/{n_detected} ({n_solved / nd:.1%})  "
          f"(bounded correction)")
    if tot_digit:
        print("  per-digit on solved cells:",
              {k: (round(per_digit_cor[k][0] / per_digit_cor[k][1], 3)
                   if per_digit_cor[k][1] else None) for k in range(1, 10)})
    for name, grid, gt, definite, solved, ok in examples:
        print("\n" + name + f" | definite acc "
              f"{(grid[definite] == gt[definite]).mean():.4f} | solved: {ok}")
        print("ground truth (candidate-only cells shown as .):")
        for r in range(9):
            print(" ".join(str(gt[r, c]) if definite[r, c] else "." for c in range(9)))
        print("recognized (wrong cells marked *):")
        marks = (grid != gt) & definite
        for r in range(9):
            print(" ".join(f"{grid[r, c]}{'*' if marks[r, c] else ' '}" for c in range(9)))


# --- primary: Lexski/sudoku-image-recognition (sealed test or sample) ---------
puzzles = []
hf_used = False
if CONFIG.get("hf_benchmark"):
    try:
        import urllib.request as _urlreq
        hf_base = ("https://huggingface.co/datasets/Lexski/sudoku-image-recognition/"
                   "resolve/main/data")
        if CONFIG.get("hf_full_test"):
            # FULL SEALED test partition: rebuilt with the same seed/algorithm
            # as photo_data.py (cell 7b), so it matches photo_splits.json.
            hf_dir = CONFIG["photo_hf_dir"]
            hf_splits = ("train", "val", "test")
            meta = {}
            for s in hf_splits:
                d = os.path.join(hf_dir, s)
                os.makedirs(d, exist_ok=True)
                p = os.path.join(d, "metadata.jsonl")
                if not os.path.exists(p):
                    _urlreq.urlretrieve(f"{hf_base}/{s}/metadata.jsonl", p)
                meta[s] = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
            name_to_split = {}
            for s, rows in meta.items():
                for row in rows:
                    name_to_split[os.path.basename(
                        row["file_name"].replace("\\", "/"))] = s
            names = sorted(name_to_split)
            rng = np.random.RandomState(CONFIG["photo_split_seed"])
            perm = rng.permutation(len(names))
            n_tr = int(round(len(names) * 0.70))
            n_va = int(round(len(names) * 0.15))
            test_names = set(names[i] for i in perm[n_tr + n_va:])
            sel_rows = []
            for s, rows in meta.items():
                for row in rows:
                    nm = os.path.basename(row["file_name"].replace("\\", "/"))
                    if nm in test_names:
                        sel_rows.append((s, nm, row))
            img_dir = os.path.join(hf_dir, "test", "images")
            os.makedirs(img_dir, exist_ok=True)
            for s, nm, _ in sel_rows:
                out = os.path.join(img_dir, nm)
                if not os.path.exists(out):
                    _urlreq.urlretrieve(f"{hf_base}/{s}/images/{nm}", out)
            for s, nm, row in sel_rows:
                out = os.path.join(img_dir, nm)
                img = cv2.imread(out)
                if img is None:
                    continue
                kp = np.array(row["keypoints"], dtype=np.float32).reshape(4, 2)
                kp = kp[[0, 3, 2, 1]]              # TL,BL,BR,TR -> TL,TR,BR,BL
                gt, definite = hf_cells_to_gt(row["cells"])
                puzzles.append({"name": nm, "img": img, "kp": kp,
                                "gt": gt, "definite": definite})
            hf_used = True
            print(f"HF benchmark: SEALED TEST partition ({len(puzzles)} puzzles, "
                  f"same names as photo_splits.json) - official number")
        else:
            hf_dir = "/content/hf_sample"
            os.makedirs(os.path.join(hf_dir, "images"), exist_ok=True)
            meta_path = os.path.join(hf_dir, "metadata.jsonl")
            if not os.path.exists(meta_path):
                _urlreq.urlretrieve(f"{hf_base}/{CONFIG['hf_split']}/metadata.jsonl",
                                    meta_path)
            rows = [json.loads(l) for l in open(meta_path, encoding="utf-8") if l.strip()]
            rng = np.random.RandomState(SEED)
            idx = rng.permutation(len(rows))[:min(CONFIG["hf_sample_size"], len(rows))]
            for i in idx:
                row = rows[i]
                name = os.path.basename(row["file_name"].replace("\\", "/"))
                out = os.path.join(hf_dir, "images", name)
                if not os.path.exists(out):
                    _urlreq.urlretrieve(f"{hf_base}/{CONFIG['hf_split']}/images/{name}",
                                        out)
                img = cv2.imread(out)
                if img is None:
                    continue
                kp = np.array(row["keypoints"], dtype=np.float32).reshape(4, 2)
                kp = kp[[0, 3, 2, 1]]              # TL,BL,BR,TR -> TL,TR,BR,BL
                gt, definite = hf_cells_to_gt(row["cells"])
                puzzles.append({"name": name, "img": img, "kp": kp,
                                "gt": gt, "definite": definite})
            hf_used = True
            print(f"HF benchmark: downloaded {len(puzzles)} images "
                  f"(seeded sample of {CONFIG['hf_sample_size']} from split "
                  f"'{CONFIG['hf_split']}' | {len(rows)} available) - lightweight")
    except Exception as exc:
        print(f"HF sample unavailable ({exc}) - falling back to the folder dataset.")

# --- fallback: wichtounet folder (version detected so V2/mixed never mix) ----
if not puzzles:
    outlines_csv = None
    for cand in glob.glob(os.path.join(CONFIG["data_dir"], "**", "outlines_sorted.csv"),
                          recursive=True):
        outlines_csv = cand
        break
    if outlines_csv:
        outlines = load_outlines(outlines_csv)
        fnames = sorted(outlines)
        rng = np.random.RandomState(SEED)
        perm = rng.permutation(len(fnames))
        n_eval = max(1, int(round(len(fnames) * CONFIG["labeled_eval_frac"])))
        eval_names = [fnames[i] for i in perm[:n_eval]]
        for fname in eval_names:
            img_path, pts = outlines[fname]
            dat_path = os.path.join(os.path.dirname(img_path),
                                    os.path.splitext(fname)[0] + ".dat")
            if not os.path.exists(dat_path):
                continue
            gt = parse_dat(dat_path)
            img = cv2.imread(img_path)
            if img is None:
                continue
            puzzles.append({"name": fname, "img": img, "kp": pts,
                            "gt": gt, "definite": None})
        version = "mixed (complete grids)" if int((gt == 0).sum()) == 0 else "V2 (givens)"
        print(f"folder benchmark: {len(puzzles)} photos "
              f"(seeded {CONFIG['labeled_eval_frac']:.0%} | version: {version})")
    else:
        print("No labeled dataset found - benchmark skipped.")

if puzzles:
    source = "HF" if hf_used else "folder"
    # both benchmark modes on the same puzzles, reported separately
    eval_puzzle_set(puzzles, source, use_detection=False)
    eval_puzzle_set(puzzles, source, use_detection=True)

# --- grid-detection-only check on unlabeled photos (e.g. cpecog.zip) ---------
ver_images = []
if CONFIG.get("cpecog_dir") and os.path.isdir(CONFIG["cpecog_dir"]):
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        ver_images += glob.glob(os.path.join(CONFIG["cpecog_dir"], "**", ext), recursive=True)

if ver_images:
    ver_images = sorted(ver_images)
    perm = np.random.RandomState(SEED).permutation(len(ver_images))
    n_eval = max(1, int(round(len(ver_images) * CONFIG["cpecog_eval_frac"])))
    eval_imgs = [ver_images[i] for i in perm[:n_eval]]
    print(f"\ngrid-detection check on {len(eval_imgs)} of {len(ver_images)} "
          f"unlabeled images (seeded {CONFIG['cpecog_eval_frac']:.0%} split)")
    # honest metric: only real detections (contour/lines) count - no resize
    # fallback, so failures are counted as failures.
    n_detected = 0
    for p in eval_imgs:
        img = cv2.imread(p)
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        warped, method = detect_and_warp(gray)
        if warped is not None:
            n_detected += 1
    print(f"grid detected:   {n_detected}/{len(eval_imgs)} "
          f"({n_detected / len(eval_imgs):.1%})")
else:
    print("(no cpecog_dir set - unlabeled detection check skipped)")
